# 75 · PT pathway methods and their specificity on the package pseudospace

**The question.** On the package units, labels and coordinate, which pathway methods are specific against specimen
noise under the swap rule, what does each call, and how far do their calls overlap with the positional F-test? Does
the species level still recover known human–mouse differences?

**Why.** The paper's method table needs one row for every pathway method it cites, judged by one rule, on one set of
units. Notebook 71 froze the primary (the positional F-test with cameraPR), the clustering comparator and the
comparison page's post hoc arms. The remaining results of record rested on SCF13, `mouse_only_v5` and notebook 13's
labels:
- notebook 66: packaged set tests on raw `T_spatial`, and the NB joint test;
- notebook 68: the level positive control;
- notebooks 31, 37, 45, 56 and 61: the conventional comparators, the coordinate-free step screen, probe exposure,
  the gene-first curve modules and the program heatmap (Figure 4e).

**The frozen protocol.** `docs/results/pt-package-pathway-methods.md` (commit 07ef30c), fixed before any fit,
including the coordinator's decisions O1–O5. Section numbers below follow the protocol's.

| Dataset | Reference | Case | Units, labels, coordinate |
|---|---|---|---|
| Human vs mouse PT | 2 control mice | 2 cortex sections of 1 human donor | package export, notebook 71's inputs |
| AKI vs control mouse PT | 2 control mice | 2 AKI mice | package export, notebook 71's inputs |

**Sections:**
1. Setup, inputs, guards; notebook 71's gene models (read, never refit).
2. One pass rule for every row.
3. The rows. 3.3, the segment F-test, runs first because notebook 74 reads its tables; then 3.1 packaged tests on raw
   `T_spatial`, 3.2 the NB joint test, 3.4 conventional pseudobulk DE, 3.5 gene-first curve modules, 3.6 the rows
   copied from notebook 71.
4. Probe exposure (human vs mouse, every row).
5. Overlap with the primary and the offset lean of the calls.
6. Programs and the Figure 4e heatmaps.
7. The level positive control (notebook 68 on the package; human vs mouse).
8. The method table, its figure, the comparison with the old numbers and the material-change rules.
9. How the protocol was read.
10. Summary.
11. Run record.

**Assumptions.** The coordinate is treated as exact. Specimens, not units, are the replicates; the two human
"replicates" are two sections of one donor. With two specimens per group every result is descriptive. Under the
package's labels the most injured AKI units leave the PT cohort (notebook 70's label checkpoint).

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
import warnings
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
external = data_root / 'external'
package_root = results_root / 'package_pseudospace'
upstream13 = results_root / 'minimal_pt_scfates'                       # the ortholog map
T71 = results_root / 'pt_pathways_package_pseudospace'                 # notebook 71: read only (tables and stage cache)
nb38 = results_root / 'pt_literature_deep'                             # probe counts; Lake cache (guard)
nb62 = results_root / 'pt_non_one_to_one_orthologs'                    # orthogroups and inventories
nb64 = results_root / 'pt_paralog_pairing_check'                       # pairs inside non-1:1 orthogroups
nb68 = results_root / 'pt_species_level_control'                       # old numbers; Lake cache (guard)
nb73_classes = results_root / 'pt_package_label_comparison' / 'tables' / '43' / 'gene_classes_symmetric.csv'
LAKE = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
CENSUS = external / 'census_pt_segments'
HOUSEKEEPING = external / 'housekeeping' / 'HK_genes.txt'
LITERATURE = project / 'docs' / 'results' / 'pt-level-literature-controls.csv'
output = results_root / 'pt_package_pathway_methods'
tables, figures = output / 'tables', output / 'figures'
level_tables = tables / 'level'
for folder in (tables, figures, level_tables):
    folder.mkdir(parents=True, exist_ok=True)
DATASETS = ['human_vs_mouse', 'aki_vs_control']
hvm, aki = DATASETS
SHORT_DS = {hvm: 'hvm', aki: 'aki'}
EXPORT = {ds: package_root / SHORT_DS[ds] / 'pt_units.h5ad' for ds in DATASETS}
NATIVE = {s: package_root / 'runs' / 'hvm' / '01_units' / f'{s}_units.h5ad' for s in ('Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1')}
FINGERPRINTS = {hvm: 'efafc9ac2e99', aki: 'bb4a80c1b484'}    # confirmed pass-1 labels (notebook 70)
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
REQUIRED = [*EXPORT.values(), *NATIVE.values(), upstream13 / 'ortholog_map_used.csv', nb38 / 'probe_counts.csv', nb73_classes,
            LAKE, HOUSEKEEPING, LITERATURE, nb64 / 'tables' / 'dropped_pairs.csv',
            *[nb62 / 'tables' / f for f in ('family_catalogue.csv', 'gene_inventory_mouse.csv', 'gene_inventory_human.csv')],
            *[CENSUS / f for f in ('mouse_pt_segment_pseudobulk_counts.csv.gz', 'mouse_pt_segment_pseudobulk_meta.csv',
                                   'human_pt_segment_pseudobulk_counts.csv.gz', 'human_pt_segment_pseudobulk_meta.csv',
                                   'mouse_pt_segment_pseudobulk_counts_families.csv.gz',
                                   'mouse_pt_segment_pseudobulk_meta_families.csv', 'mouse_family_genes_fetched.csv',
                                   'census_symbol_audit.csv')],
            *[T71 / 'tables' / f'{kind}_{ds}.{ext}' for ds in DATASETS
              for kind, ext in (('universe', 'csv'), ('gene_sets', 'json'), ('gene_statistics', 'csv'),
                                ('redundancy_clusters', 'csv'), ('s1_pathway_tests', 'csv'), ('s1_gene_z', 'csv'),
                                ('s2_ssgsea_tests', 'csv'))],
            *[T71 / 'tables' / f for f in ('inputs.csv', 'method_table.csv', 's1_pass_table.csv', 's2_ssgsea_pass_table.csv',
                                           's3_level_variants_summary.csv', 's3_level_gene_table.csv')],
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in LIBRARIES]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 38, 62, 64, 70, 71 and 73 first)')

NOTEBOOK_LOGIC_VERSION = '75.package.1'   # Bump when a cached calculation changes.
MT_PREFIX = 'mt-'
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05      # notebook 31's relabeling rule
CALIBRATION_MAX = 1.5 * ALPHA                    # notebook 61's criterion 2
CAMERA_COR = .01                                 # limma's default inter.gene.cor
BASIS_DF, GRID_POINTS, DF1 = 6, 101, 6
DF2 = {'condition': 12, 'relabeling': 6}         # notebook 71's positional F-test (guard)
SEGMENT_DF = {'df1': 2, 'condition': 4, 'relabeling': 2}   # the segment F-test (asserted from the design ranks)
SET_SIZES = (10, 300)
GSEA_PERMUTATIONS, GSEA_SEED = 1000, 12
N_NULL, MATCH_SEED = 9999, 12                    # the NB joint test (notebook 66)
LFC = 1.                                         # DE + ORA lists: padj <= 0.05 and |log2FC| >= 1 (notebook 56)
TOP_SHARE, K_RANGE, N_RESAMPLE, KM_SEED, KM_STARTS = .10, range(3, 9), 20, 56, 20   # curve modules (notebooks 56, 61)
MIN_PER_MODULE = 10                              # notebook 61: a relabeling list with < 10 k genes has no modules
EXPOSED = .5                                     # probe-exposed when >= half of the calls are lost on equal-probe genes
PROGRAM_CUT, MAX_ROWS, HEAT_BINS, Z_CLIP = .5, 40, 12, 2.5   # programs and heatmaps (notebook 61)
M7_LIMIT = .3
EXTERNAL_COMMIT = '59da04b'
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))     # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import distribution, version
from IPython.display import display

direct_url = json.loads(distribution('pseudospace-reconstruction').read_text('direct_url.json') or '{}')
EXTERNAL = {'version': version('pseudospace-reconstruction'),
            'commit': direct_url.get('vcs_info', {}).get('commit_id', 'unknown'), 'url': direct_url.get('url', '')}
assert EXTERNAL['commit'].startswith(EXTERNAL_COMMIT), f'pseudospace_reconstruction is not commit {EXTERNAL_COMMIT}: {EXTERNAL}'

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
FAMILY_COLOR = {'positional': '#1b8a5a', 'discrete': '#2a78d6', 'average-level': '#eb6834', 'gene-first': '#eda100',
                'score': '#9C93D3'}
REFERENCE_COLOR, CASE_COLOR = '#0072B2', '#D55E00'
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


print('Results folder:', output, '· parallel workers:', N_JOBS, '· pseudospace_reconstruction', EXTERNAL['version'],
      EXTERNAL['commit'][:7])

### 1.1 · Inputs and guards

The inputs are built exactly as notebook 71 builds them (`package_inputs.hvm_inputs` and `aki_inputs`, mt genes out
of the library and the universe, the common support, position min–max over it, the 2% detection rule, the 10–300 gene
sets and the 3-bin matching strata).

**Guards (not cached):**
- G1 (= 71's G1–G3): the export contract; provenance notebook 70 at package 59da04b with the confirmed pass-1
  fingerprints (human vs mouse `efafc9ac2e99`, AKI `bb4a80c1b484`); the 13 ortholog-map `mt-` genes leave the
  library and the universe and nothing else moves.
- G2: the rebuilt inputs are 71's: genes equal `universe_{ds}.csv`, sets equal `gene_sets_{ds}.json`, unit counts
  equal `inputs.csv`.

In [ ]:
import anndata as ad
from scipy import sparse

from pseudospace.package_inputs import aki_inputs, hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import common_grid, partition_groups
from pseudospace.stage_cache import cached_payload, digest

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith(MT_PREFIX))
assert len(MT) == 13, MT
UNITS = {ds: read_pt_units(EXPORT[ds], SHORT_DS[ds]) for ds in DATASETS}
guard_log, PROVENANCE = [], {}
for ds, units in UNITS.items():
    prov = units.uns['provenance']
    assert 'dev only' not in json.dumps(prov, default=str), 'the ws71 development object is not an export'
    assert str(prov['notebook_logic_version']).startswith('70.'), prov['notebook_logic_version']
    assert str(prov['package']['commit']).startswith(EXTERNAL_COMMIT), prov['package']
    fingerprint = prov['fingerprints']['pass1']['membership_sha1']
    assert fingerprint == FINGERPRINTS[ds], (ds, fingerprint)
    PROVENANCE[ds] = {'notebook': str(prov['notebook']), 'logic': str(prov['notebook_logic_version']),
                      'package_commit': str(prov['package']['commit']), 'pass1_fingerprint': fingerprint,
                      'units': int(units.n_obs), 'genes': int(units.n_vars)}
guard_log.append('G1: export contract; provenance notebook 70 at package 59da04b with the confirmed pass-1 fingerprints')

kept_h = hvm_inputs(UNITS[hvm], data_root, orthologs)
built = {hvm: hvm_inputs(UNITS[hvm], data_root, orthologs, exclude_genes=MT)}
built[aki] = aki_inputs(UNITS[aki], data_root, orthologs, kept_h['genes'], exclude_genes=MT)
kept_a = aki_inputs(UNITS[aki], data_root, orthologs, kept_h['genes'])
KEPT = {hvm: kept_h, aki: kept_a}
TITLES = {hvm: ('Human vs mouse PT', 'human', 'mouse'), aki: ('AKI vs control mouse PT', 'AKI', 'control')}
INPUTS71 = pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0)
DATA = {}
for ds in DATASETS:
    new, kept = built[ds], KEPT[ds]
    assert new['genes'] == [g for g in kept['genes'] if g not in MT]                                      # G1 (71's G3)
    assert set(new['excluded_library']) <= set(MT) and set(new['excluded_universe']) <= set(MT)
    names = UNITS[ds].var_names
    mask = UNITS[ds].obs_names.isin(new['obs'].index)
    mt_counts = np.asarray(UNITS[ds].X[mask][:, names.get_indexer(new['excluded_library'])].sum(axis=1)).ravel()
    np.testing.assert_allclose(new['library'], kept['library'] - mt_counts, rtol=0, atol=1e-6)
    title, case_name, reference_name = TITLES[ds]
    universe71 = pd.read_csv(T71 / 'tables' / f'universe_{ds}.csv').gene.astype(str).tolist()
    sets71 = json.loads((T71 / 'tables' / f'gene_sets_{ds}.json').read_text())
    assert new['genes'] == universe71, 'G2: the universe differs from notebook 71'                         # G2
    assert new['gene_sets'] == sets71, 'G2: the gene sets differ from notebook 71'
    assert int(INPUTS71.loc['units in the common support', title]) == len(new['position'])
    medians = pd.DataFrame({'p': new['position'], 's': new['specimen'], 'g': new['segment']}).groupby(['s', 'g']).p.median().unstack()
    assert (medians['PT-S1'] < medians['PT-S2']).all() and (medians['PT-S2'] < medians['PT-S3']).all()
    DATA[ds] = {
        'title': title, 'case_name': case_name, 'reference_name': reference_name,
        'counts': new['counts'].tocsr(), 'library': np.asarray(new['library'], float),
        'position': np.asarray(new['position'], float), 'specimen': new['specimen'].astype(str),
        'segment': pd.Series(new['segment']).str.replace('PT-', '', regex=False).to_numpy(),
        'case': np.asarray(new['case'], bool), 'genes': list(new['genes']), 'sets': new['gene_sets'],
        'obs': new['obs'], 'covariates': new['covariates'], 'strata': np.asarray(new['strata']),
        'n_units': new['n_units']}
    guard_log.append(f'{title}: G1 mt genes out of the library and universe; G2 genes, sets and units equal notebook 71')
del kept_h, kept_a, KEPT, built

PARTITIONS, SHORT = {}, {}


def slug(label):
    return label.replace(':', '_').replace('+', '_')


def group_one(ds, label):
    d = DATA[ds]
    return sorted(set(d['specimen'][PARTITIONS[ds][label][0] == 1]))


for ds, d in DATA.items():
    PARTITIONS[ds] = partition_groups(d['specimen'], d['case'])
    labels = list(PARTITIONS[ds])
    assert labels[0] == 'species'
    SHORT[ds] = dict(zip(labels, ['condition', 'relabeling 1', 'relabeling 2']))
    d['input_key'] = {'counts': digest(d['counts']), 'library': d['library'], 'position': d['position'],
                      'specimen': d['specimen'], 'genes': d['genes']}
    d['log_mean_count'] = np.log(np.asarray(d['counts'].mean(axis=0)).ravel())    # notebook 67's squeezeVar covariate
    d['case_specimens'] = sorted(set(d['specimen'][d['case']]))
    d['reference_specimens'] = sorted(set(d['specimen'][~d['case']]))
INPUT_TABLE = pd.DataFrame({d['title']: {
    'units (common support)': len(d['position']), 'genes': len(d['genes']), 'pathways': len(d['sets']),
    'reference specimens': ', '.join(d['reference_specimens']), 'case specimens': ', '.join(d['case_specimens']),
    **{f'{SHORT[ds][label]} (one side)': ' + '.join(group_one(ds, label)) for label in list(PARTITIONS[ds])[1:]},
    'units per segment': ', '.join(f"{s} {int((d['segment'] == s).sum()):,}" for s in ('S1', 'S2', 'S3')),
    'pass-1 fingerprint': PROVENANCE[ds]['pass1_fingerprint']} for ds, d in DATA.items()})
INPUT_TABLE.to_csv(tables / 'inputs.csv')
display(INPUT_TABLE)
print('\n'.join('· ' + line for line in guard_log))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

### 1.2 · Redundancy clusters and the package's gene-set object

Notebook 71's redundancy clusters are read, not rebuilt. The package's gene-set object (`build_gene_sets`) is built on
71's sets for the clustering method's ORA in section 4; it must hold exactly 71's sets and members (71's G6).

In [ ]:
from pseudospace_reconstruction.compare.pathways import PathwayParams, ora_by_group
from pseudospace_reconstruction.io.gene_sets import build_gene_sets

REDUNDANCY, EXT_SETS = {}, {}
for ds, d in DATA.items():
    REDUNDANCY[ds] = pd.read_csv(T71 / 'tables' / f'redundancy_clusters_{ds}.csv', index_col=0).reindex(list(d['sets']))
    assert REDUNDANCY[ds].representative.notna().all()
    sets = build_gene_sets({lib: {p.split('::', 1)[1]: v for p, v in d['sets'].items() if p.startswith(lib + '::')}
                            for lib in LIBRARIES}, d['genes'], min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
    ids = [f'{lib}::{name}' for lib, name in zip(sets.library, sets.pathway)]
    assert sorted(ids) == sorted(d['sets']) and all(set(sets.member_names(i)) == set(d['sets'][p]) for i, p in enumerate(ids))
    EXT_SETS[ds] = sets
guard_log.append("the package's gene-set builder and notebook 71's sets agree, both datasets")


def representatives(ds, calls):
    """The redundancy clusters a call set touches, named by their medoids."""
    return set(REDUNDANCY[ds].representative.reindex(sorted(calls)))


NAMES = {ds: pd.Series({p: p.split('::', 1)[1] for p in DATA[ds]['sets']}).str.replace(r' R-HSA-\d+$', '', regex=True)
         for ds in DATASETS}

### 1.3 · Notebook 71's gene models and statistics (read, never refit)

Every partition's NB gene model (`gene_model_{ds}_{label}`: `T_spatial`, flags, α, Δ(x), the grid and the specimen
curves) is read from notebook 71's stage cache, and its positional F-test (moderated F, z, q) from
`gene_statistics_{ds}.csv`.

**Guard G3 (not cached):** each payload is 71's run: its genes and `T_spatial` equal the gene table (rtol 1e-12), its
flags are within the table's, and the moderated F recomputed from the table's `T_spatial` and `LR_spec` reproduces z.
A purged cache stops the notebook; it never refits.

In [ ]:
from scipy.stats import false_discovery_control, norm

from pseudospace import discrete_comparators, nb_gam, pathway_calibration, pathway_pipelines, stats_gam
from pseudospace.pathway_pipelines import GeneModel, camera_pr, moderated_f, rank_first_gsea, rank_normal_scores

cache = output / 'stage_cache'
code_key = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in (nb_gam, pathway_calibration, pathway_pipelines,
                                                                          stats_gam, discrete_comparators)])
SECONDS = {}


def cached(stage, compute, *, params=None, inputs=None, key=None):
    """cached_payload under this notebook's root and code; keeps the stage's uncached compute time."""
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=key or code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


MODELS, STAT71 = {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}
for ds, d in DATA.items():
    table = pd.read_csv(T71 / 'tables' / f'gene_statistics_{ds}.csv', header=[0, 1], index_col=0)
    assert table.index.astype(str).tolist() == d['genes']
    for label in PARTITIONS[ds]:
        model = model_from(stored(T71, f'gene_model_{ds}_{slug(label)}'), d['genes'])
        frame = table[SHORT[ds][label]].copy()
        frame[['converged', 'separated']] = frame[['converged', 'separated']].astype(bool)
        np.testing.assert_allclose(model.statistics.T_spatial.to_numpy(), frame.T_spatial.to_numpy(), rtol=1e-12, atol=0)  # G3
        assert not (model.statistics.separated & ~frame.separated).any() and not (frame.converged & ~model.statistics.converged).any()
        df2 = DF2['condition' if label == 'species' else 'relabeling']
        again = moderated_f(frame.T_spatial, DF1, frame.LR_spec.to_numpy(), df2, covariate=d['log_mean_count'], robust=False)
        np.testing.assert_allclose(again.z.to_numpy(), frame.z.to_numpy(), rtol=1e-9, atol=1e-9)
        np.testing.assert_allclose(model.grid, common_grid(d['position'], d['specimen'], GRID_POINTS), rtol=0, atol=1e-12)
        MODELS[ds][label], STAT71[ds][label] = model, frame
guard_log.append("G3: notebook 71's six gene models read from its cache; T_spatial, flags and the moderated F reproduce its tables")
print(guard_log[-1])

## 2 · One pass rule for every row

Every row, on each dataset where it is defined, is judged by one rule (`discrete_comparators.swap_rule`), including
the rows copied from 71 and the conventional DE rows. Each row has three runs: the condition split and the two
balanced relabelings (`partition_groups`). A row **passes** when all of these hold:
- it makes calls (> 0) on the condition split;
- NCDR < 0.25 (mean relabeled calls ÷ condition calls);
- each relabeling calls < 5% of the tested sets;
- calibration share ≤ 0.075: the share of relabeled per-set calibration p ≤ 0.05, pooled over both relabelings.

What differs by row is only how each test's per-set p and calls are read:
- one-sided set tests (cameraPR Up calls, GSEA NES > 0 calls, the joint test): the one-sided p;
- signed set tests (calls in either direction): the test's own p (cameraPR two-sided p; GSEA nominal p);
- several tests per set (S1/S2/S3 segments, ORA lists, modules, clustering lists): the smallest p × the number of
  tests, capped at 1.

Gene universe and sets are 71's for every row; a set is tested where 10–300 of its members carry the row's statistic.

In [ ]:
from pseudospace.discrete_comparators import family_ora, membership, segment_f_test, swap_rule

ROWS, CALLS, CALIB = [], {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}
ROW_INFO = {}


def register(ds, key, label, family, source, model, calls, calibration, tested, returns, condition_runs=None):
    """One method-table row. calls: {partition label: set of called sets}; calibration: {relabeling: per-set p}.
    condition_runs: several condition call sets (the clustering method's single-split null) instead of calls['species']."""
    labels = list(PARTITIONS[ds])
    runs = condition_runs if condition_runs is not None else [calls[labels[0]]]
    relabeled = [calls[label] for label in labels[1:]]
    p = np.concatenate([np.asarray(calibration[label], float) for label in labels[1:]])
    verdict = swap_rule([len(r) for r in runs], [len(r) for r in relabeled], tested, p)
    ROWS.append({'dataset': DATA[ds]['title'], 'row': key, 'method': label, 'family': family, 'source': source,
                 'gene model': model, 'calls': ', '.join(str(len(r)) for r in runs),
                 'calls (redundancy clusters)': ', '.join(str(len(representatives(ds, r))) for r in runs),
                 'relabeled calls': ', '.join(str(len(r)) for r in relabeled), 'tested': int(tested),
                 'NCDR': verdict['NCDR'], 'relabeling rule': verdict['relabeling rule'],
                 'calibration share': verdict['calibration share'], 'calibration': verdict['calibration'],
                 'pass': verdict['pass'], 'returns': returns})
    CALLS[ds][key] = {'condition': runs, **{label: calls[label] for label in labels[1:]}}
    CALIB[ds][key] = p
    ROW_INFO[key] = {'label': label, 'family': family}


def testable(sets, genes):
    """71's sets that keep 10-300 members among ``genes``."""
    keep = set(genes)
    return [p for p, v in sets.items() if SET_SIZES[0] <= len(keep.intersection(v)) <= SET_SIZES[1]]

## 3.3 · The coordinate-free step screen: the segment F-test (runs first; notebook 74 reads its tables)

The segment twin of the primary: everything is the positional F-test except the basis, which is the package's
S1/S2/S3 labels instead of the spline (`discrete_comparators.segment_f_test`).
- Rows: notebook 71's units; NB GLM on raw counts, offset log(mt-free library), weights n / (2·J·n_j).
- Designs: segment intercepts as the shared shape, within-group specimen contrasts, and in a relabeling species ×
  segment as nuisance; `level` adds the group, `steps` adds group × segment, `spec` adds within-group specimen
  contrasts × segment.
- `T_step` = LR(level → steps), df₁ 2; `LR_spec` = LR(steps → spec), df₂ 4 (condition) or 2 (relabeling), asserted.
- α: notebook 71's condition-split α for every partition; no dispersion is estimated.
- squeezeVar with log mean count (robust FALSE), F = (T_step/2)/s²_post, z = Φ⁻¹(1 − p), gene q by BH; cameraPR on z
  (0.01); a call is Up with FDR ≤ 0.05.
- Per-segment group differences Δ_S1–Δ_S3 (case − reference, natural log, equal specimen weights) with HC3 SEs.

**Saved under stable names** (protocol section 8): `gene_statistics_segment_{ds}.csv` and
`pathway_tests_segment_{ds}.csv`, column blocks `condition`, `relabeling 1`, `relabeling 2`. The joint-test
sensitivity (`pathway_tests_segment_joint_{ds}.csv`) follows in 3.2.

In [ ]:
SEGMENT_COLUMNS = ['s2', 's2_prior', 'd0', 's2_post', 'F', 'df_total', 'log_p', 'p', 'z', 'q', 'T_step', 'LR_spec', 'df2',
                   'converged', 'separated', 'delta_S1', 'delta_S2', 'delta_S3', 'se_S1', 'se_S2', 'se_S3']
SEG, SEG_TESTS = {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}
t0 = time.perf_counter()
for ds, d in DATA.items():
    alpha = MODELS[ds]['species'].dispersion
    for label, (group, nuisance) in PARTITIONS[ds].items():
        payload = cached(f'segment_ftest_{ds}_{slug(label)}',
                         lambda d=d, group=group, nuisance=nuisance, alpha=alpha: {
                             c: v.to_numpy() for c, v in segment_f_test(
                                 d['counts'], d['library'], d['segment'], group, d['specimen'], alpha, genes=d['genes'],
                                 nuisance=nuisance, covariate=d['log_mean_count'], robust=False, n_jobs=N_JOBS).items()},
                         params={'label': label, 'robust': False, 'covariate': 'log mean count'},
                         inputs={**d['input_key'], 'segment': d['segment'], 'group': group, 'nuisance': nuisance,
                                 'alpha': alpha['alpha'], 'alpha_converged': alpha['converged']})
        frame = pd.DataFrame({c: payload[c] for c in SEGMENT_COLUMNS}, index=pd.Index(d['genes'], name='gene'))
        frame[['converged', 'separated']] = frame[['converged', 'separated']].astype(bool)
        expected = SEGMENT_DF['condition' if label == 'species' else 'relabeling']
        assert (frame.df2 == expected).all(), (ds, label, frame.df2.iloc[0])
        assert np.isfinite(frame.z).all() and np.isfinite(frame.F).all()
        SEG[ds][label] = frame
        test = camera_pr(frame.z, d['sets'], inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)
        assert list(test.pathway_id) == list(d['sets'])
        SEG_TESTS[ds][label] = test.set_index('pathway_id')
    pd.concat({SHORT[ds][label]: SEG[ds][label] for label in PARTITIONS[ds]}, axis=1).to_csv(
        tables / f'gene_statistics_segment_{ds}.csv')
    pd.concat({SHORT[ds][label]: SEG_TESTS[ds][label][['n', 'direction', 'p', 'fdr', 'p_one_sided', 'call']]
               for label in PARTITIONS[ds]}, axis=1).to_csv(tables / f'pathway_tests_segment_{ds}.csv')
    labels = list(PARTITIONS[ds])
    register(ds, 'segment', 'segment F-test · cameraPR on z (coordinate-free S1/S2/S3 steps)', 'discrete', '75',
             'NB, segment steps', {l: set(SEG_TESTS[ds][l].index[SEG_TESTS[ds][l].call.astype(bool)]) for l in labels},
             {l: SEG_TESTS[ds][l].p_one_sided.to_numpy(float) for l in labels[1:]}, len(d['sets']),
             'per set: p, FDR; per gene: F, z, q and the step in each segment')
SEGMENT_SUMMARY = pd.DataFrame([{
    'dataset': DATA[ds]['title'], 'partition': SHORT[ds][label], 'df2': int(SEG[ds][label].df2.iloc[0]),
    'd0 (median)': float(SEG[ds][label].d0.median()), 'median F': float(SEG[ds][label].F.median()),
    'genes p ≤ 0.05': int((SEG[ds][label].p <= .05).sum()), 'genes BH ≤ 0.05': int((SEG[ds][label].q <= .05).sum()),
    'non-converged (median-filled)': int((~SEG[ds][label].converged).sum()), 'separated': int(SEG[ds][label].separated.sum()),
    'pathway calls': int(SEG_TESTS[ds][label].call.sum()),
    'seconds': round(SECONDS[f'segment_ftest_{ds}_{slug(label)}'], 1)} for ds in DATASETS for label in PARTITIONS[ds]])
SEGMENT_SUMMARY.to_csv(tables / 'segment_ftest_summary.csv', index=False)
display(SEGMENT_SUMMARY.round(3))
display(pd.DataFrame(ROWS).set_index(['dataset', 'method']).drop(columns='returns').round(3))
print(f'Segment F-test {time.perf_counter() - t0:.0f} s; tables written for notebook 74')

## 3.1 · Packaged tests on raw `T_spatial` (notebook 66, method A)

`T_spatial` of each partition is read from notebook 71's gene models.
- **Primary:** limma `cameraPR` on rank-normal scores Φ⁻¹((rank − 0.5)/G) (`inter.gene.cor` 0.01, `use.ranks` FALSE); a
  call is Up with FDR ≤ 0.05.
- **Sensitivities:** `use.ranks` TRUE; gseapy preranked GSEA on the rank-normal scores and on raw `T_spatial` (1,000
  permutations, seed 12, sizes 10–300; a call is NES > 0 and FDR q ≤ 0.05).
- **Diagnostics** (notebook 66's section 5.2, now pre-specified): the inflation a and ρ implied by the relabeled
  cameraPR z (`decoy_inflation`, E[z²] = a(1 + (m − 1)ρ)); cameraPR on scores rank-normalised within the 27 matching
  strata; the Spearman of `T_spatial` with mean expression; the share of the score variance between strata.

In [ ]:
from scipy.stats import rankdata, spearmanr

from pseudospace.pathway_decoys import decoy_inflation, joint_matched_test


def gsea_payload(statistic, sets, metric):
    t = rank_first_gsea(statistic, sets, metric=metric, min_size=SET_SIZES[0], max_size=SET_SIZES[1],
                        permutation_num=GSEA_PERMUTATIONS, seed=GSEA_SEED, threads=N_JOBS, fdr=ALPHA)
    return {'pathway_id': t.pathway_id.to_numpy().astype(str), 'NES': t.NES.to_numpy(float), 'p': t.p.to_numpy(float),
            'q': t.q.to_numpy(float), 'p_one_sided': t.p_one_sided.to_numpy(float), 'call': t.call.to_numpy(bool)}


def gsea_frame(g):
    return pd.DataFrame({k: g[k] for k in ('NES', 'p', 'q', 'p_one_sided', 'call')}, index=pd.Index(g['pathway_id'].astype(str), name='pathway_id'))


def within_stratum_scores(statistic, strata):
    """Rank-normal scores computed separately inside each matching stratum (notebook 66)."""
    values = pd.Series(np.asarray(statistic, float), index=statistic.index)
    return values.groupby(np.asarray(strata)).transform(lambda v: pd.Series(norm.ppf((rankdata(v) - .5) / len(v)), index=v.index))


RAW_KEYS = {'raw_camerapr': 'cameraPR on rank-normal raw T_spatial (66)', 'raw_camerapr_ranks': 'cameraPR, use.ranks = TRUE, raw T_spatial (66)',
            'raw_gsea_ranknormal': 'GSEA on rank-normal raw T_spatial (66)', 'raw_gsea_raw': 'GSEA on raw T_spatial (66)'}
RAW = {ds: {k: {} for k in RAW_KEYS} for ds in DATASETS}
DIAG_ROWS = []
t0 = time.perf_counter()
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    for label, model in MODELS[ds].items():
        t_spatial = model.statistics.T_spatial
        score = rank_normal_scores(t_spatial)
        for key, ranks in (('raw_camerapr', False), ('raw_camerapr_ranks', True)):
            RAW[ds][key][label] = camera_pr(score, d['sets'], inter_gene_cor=CAMERA_COR, use_ranks=ranks, fdr=ALPHA).set_index('pathway_id')
        for key, metric in (('raw_gsea_ranknormal', 'rank_normal'), ('raw_gsea_raw', 'raw')):
            g = cached(f'raw_gsea_{ds}_{metric}_{slug(label)}', lambda t_spatial=t_spatial, metric=metric, d=d: gsea_payload(t_spatial, d['sets'], metric),
                       params={'permutation_num': GSEA_PERMUTATIONS, 'seed': GSEA_SEED, 'sizes': SET_SIZES, 'metric': metric},
                       inputs={'T_spatial': t_spatial.to_numpy(), 'genes': d['genes'], 'sets': d['sets']})
            RAW[ds][key][label] = gsea_frame(g)
    for key, label_text in RAW_KEYS.items():
        frames = RAW[ds][key]
        register(ds, key, label_text, 'positional', '75 (66, method A)', 'NB, spline',
                 {l: set(frames[l].index[frames[l].call.astype(bool)]) for l in labels},
                 {l: frames[l].p_one_sided.fillna(1.).to_numpy(float) for l in labels[1:]}, len(frames[labels[0]]),
                 'per set: direction or NES, p, FDR')
    # diagnostics
    sizes = pd.Series({p: len(v) for p, v in d['sets'].items()})
    gene_rows = {}
    within = {}
    for label, model in MODELS[ds].items():
        score = rank_normal_scores(model.statistics.T_spatial)
        between = score.groupby(d['strata']).transform('mean')
        gene_rows[label] = (spearmanr(model.statistics.T_spatial, d['covariates'].mean_expression.reindex(d['genes']))[0],
                            float(between.var() / score.var()))
        within[label] = camera_pr(within_stratum_scores(model.statistics.T_spatial, d['strata']), d['sets'],
                                  inter_gene_cor=CAMERA_COR, fdr=ALPHA).set_index('pathway_id')
    for name, tests in (('cameraPR on rank-normal raw T_spatial', RAW[ds]['raw_camerapr']),
                        ('cameraPR, within-stratum scores', within)):
        z = pd.DataFrame({r: tests[r].p_one_sided.clip(1e-300, 1 - 1e-16).map(norm.isf) for r in labels[1:]})
        a, rho, _ = decoy_inflation(z, sizes)
        calls = [int(tests[p].call.sum()) for p in labels]
        share = float(np.mean(np.concatenate([tests[r].p_one_sided.to_numpy(float) for r in labels[1:]]) <= ALPHA))
        DIAG_ROWS.append({'dataset': d['title'], 'test': name, 'calls': calls[0], 'relabeled calls': f'{calls[1]}, {calls[2]}',
                          'NCDR': np.mean(calls[1:]) / calls[0] if calls[0] else np.nan, 'calibration share': share,
                          'implied a': a, 'implied ρ': rho,
                          **{f'Spearman T_spatial vs mean expression · {SHORT[ds][p]}': gene_rows[p][0] for p in labels},
                          **{f'score variance between strata · {SHORT[ds][p]}': gene_rows[p][1] for p in labels}})
DIAGNOSTICS = pd.DataFrame(DIAG_ROWS)
DIAGNOSTICS.to_csv(tables / 'diagnostics_raw_tspatial.csv', index=False)
for ds in DATASETS:
    pd.concat({f'{key} · {SHORT[ds][l]}': RAW[ds][key][l] for key in RAW_KEYS for l in PARTITIONS[ds]}, axis=1).to_csv(
        tables / f'pathway_tests_raw_tspatial_{ds}.csv')
display(DIAGNOSTICS.set_index(['dataset', 'test']).round(3))
print(f'3.1: {time.perf_counter() - t0:.0f} s')

## 3.2 · The NB joint test (notebook 66's reference row), and the joint-test sensitivity of the segment F-test

`matched_pathway_tests` on the gene statistic (a rank-AUC against 9,999 random sets matched within the builder's 27
strata, seed 12), `normal_matched_q`, the variance inflation fitted on the two relabelings' matched z
(`decoy_inflation`), then `joint_matched_test`; a call is effect > 0 and q ≤ 0.05. The cross-fitted calibration share
(VIF fitted on one relabeling, judged on the other) is reported beside it. Notebook 66's guard a ≈ 1.04 is dropped:
it was specific to SCF13.
- On `T_spatial`: the NB joint test row.
- On `T_step`: the segment F-test's sensitivity (the old anchor's set test, on NB).

In [ ]:
from joblib import Parallel, delayed, parallel_config
from threadpoolctl import threadpool_limits

from pseudospace.pathway_calibration import normal_matched_q
from pseudospace.pathway_remodeling import matched_pathway_tests

MATCHED_COLUMNS = ('n_genes', 'auc', 'effect', 'null_auc_mean', 'null_auc_sd')


def _matched_one(frame, sets, strata):
    with threadpool_limits(1):
        return matched_pathway_tests(frame, sets, strata, n_null=N_NULL, seed=MATCH_SEED)


def restrict_sets(sets, keep):
    """Notebook 56's restriction: members in ``keep``; sets with 10-300 of them (and fewer than all genes)."""
    keep = set(keep)
    out = {p: [g for g in v if g in keep] for p, v in sets.items()}
    return {p: v for p, v in out.items() if SET_SIZES[0] <= len(v) <= min(SET_SIZES[1], len(keep) - 1)}


def joint_test(ds, stage, statistic, frames, keep=None):
    """The NB joint test of one gene statistic on all partitions. frames: {label: Series over the genes}; keep: a gene
    subset (section 4). Returns ({label: per-set frame}, fit summary)."""
    d = DATA[ds]
    labels, swaps = list(PARTITIONS[ds]), list(PARTITIONS[ds])[1:]
    genes = pd.Index(d['genes'])
    rows = np.ones(len(genes), bool) if keep is None else genes.isin(sorted(keep))
    sets = d['sets'] if keep is None else restrict_sets(d['sets'], genes[rows])
    strata = d['strata'][rows]
    scores = {l: pd.DataFrame({statistic: np.asarray(frames[l], float)[rows]}, index=genes[rows]) for l in labels}

    def matched_all():
        with parallel_config(backend='loky', inner_max_num_threads=1):
            parts = Parallel(n_jobs=len(labels))(delayed(_matched_one)(scores[l], sets, strata) for l in labels)
        table = pd.concat([t.assign(partition=l) for l, t in zip(labels, parts)], ignore_index=True)
        return {**{c: table[c].to_numpy().astype(str) for c in ('pathway_id', 'partition')},
                **{c: table[c].to_numpy(float) for c in MATCHED_COLUMNS}}
    got = cached(stage, matched_all, params={'n_null': N_NULL, 'seed': MATCH_SEED, 'statistic': statistic},
                 inputs={'scores': {l: scores[l][statistic].to_numpy() for l in labels}, 'genes': list(genes[rows]),
                         'sets': sets, 'strata': strata})
    matched = normal_matched_q(pd.DataFrame({k: (v.astype(str) if v.dtype.kind in 'OU' else v) for k, v in got.items()})
                               .assign(statistic=statistic))
    null_z = pd.DataFrame({p: matched[matched.partition.eq(p)].set_index('pathway_id').z_matched.reindex(list(sets)) for p in swaps})
    sizes = pd.Series({p: len(v) for p, v in sets.items()})
    a, rho, vif = decoy_inflation(null_z, sizes)
    joint = joint_matched_test(matched.drop(columns=['z_matched']), vif)
    out = {}
    for label in labels:
        r = joint[joint.partition.eq(label)].set_index('pathway_id').reindex(list(sets))
        out[label] = pd.DataFrame({'effect': r.effect.to_numpy(), 'z_joint': r.z_joint.to_numpy(), 'p_one_sided': r.p_joint.to_numpy(),
                                   'q': r.q_joint.to_numpy(), 'call': (r.effect.gt(0) & r.q_joint.le(ALPHA)).to_numpy()},
                                  index=pd.Index(list(sets), name='pathway_id'))
    cross = []
    for fit_on, judge in ((swaps[0], swaps[1]), (swaps[1], swaps[0])):
        _, _, vif_one = decoy_inflation(null_z[[fit_on]], sizes)
        cross.append(norm.sf(null_z[judge] / np.sqrt(vif_one.reindex(null_z.index))))
    fit = {'a': float(a), 'rho': float(rho), 'median VIF': float(vif.median()),
           'cross-fitted calibration share': float(np.mean(np.concatenate(cross) <= ALPHA))}
    return out, fit


JOINT, JOINT_FIT = {ds: {} for ds in DATASETS}, []
t0 = time.perf_counter()
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    for key, statistic, frames, label_text, family in (
            ('nb_joint', 'T_spatial', {l: MODELS[ds][l].statistics.T_spatial.to_numpy() for l in labels},
             'NB joint test on T_spatial, relabeling-fitted VIF (66)', 'positional'),
            ('segment_joint', 'T_step', {l: SEG[ds][l].T_step.to_numpy() for l in labels},
             'segment steps: NB joint test on T_step (sensitivity)', 'discrete')):
        JOINT[ds][key], fit = joint_test(ds, f'matched_{key}_{ds}', statistic, frames)
        JOINT_FIT.append({'dataset': d['title'], 'row': key, **fit})
        res = JOINT[ds][key]
        register(ds, key, label_text, family, '75 (66, reference row)' if key == 'nb_joint' else '75', 'NB, spline' if key == 'nb_joint' else 'NB, segment steps',
                 {l: set(res[l].index[res[l].call.astype(bool)]) for l in labels},
                 {l: res[l].p_one_sided.to_numpy(float) for l in labels[1:]}, len(res[labels[0]]),
                 'per set: matched AUC, z, p, q')
    pd.concat({SHORT[ds][l]: JOINT[ds]['segment_joint'][l] for l in labels}, axis=1).to_csv(tables / f'pathway_tests_segment_joint_{ds}.csv')
    pd.concat({SHORT[ds][l]: JOINT[ds]['nb_joint'][l] for l in labels}, axis=1).to_csv(tables / f'pathway_tests_nb_joint_{ds}.csv')
JOINT_FIT = pd.DataFrame(JOINT_FIT)
JOINT_FIT.to_csv(tables / 'joint_test_vif_fits.csv', index=False)
display(JOINT_FIT.round(4))
print(f'3.2: {time.perf_counter() - t0:.0f} s')

## 3.4 · Conventional pseudobulk DE (notebook 31's code; DESeq2 keeps its own model)

- **Pseudobulks:** raw counts summed per specimen × package segment over notebook 71's units; whole PT = S1 + S2 + S3.
  The mt genes are not in the matrix, so DESeq2's size factors are mt-free.
- **Genes:** notebook 71's universe (O3). In a scope a gene whose four pseudobulks total zero has no Wald stat and is
  untested there (counted below).
- **DESeq2:** `conventional_pt.deseq2_contrast` (PyDESeq2, Wald, `refit_cooks=False`) per scope (whole PT, S1, S2,
  S3). Condition split `~condition` (2 residual df); relabeling `~condition + group`, contrast group (1 residual df).
- **Signed GSEA** (31's `signed_gsea`): `gp.prerank` on the Wald stat, weight 1, `method='multilevel'`, seed 12,
  10–300. BH of the nominal p per partition over sets (whole PT) or set × segment (S1/S2/S3); a call is q ≤ 0.05 in
  either sign. Calibration p: the nominal p (whole PT); the smallest segment p × the number of segments tested.
- **ORA** (56's rule, `discrete_comparators.family_ora`): lists are padj ≤ 0.05 and |log2FC| ≥ 1, one per direction
  (whole PT 2 lists; S1/S2/S3 6); one-sided hypergeometric against the genes with a Wald stat in that scope; BH over
  set × list within the partition; a call is any test with q ≤ 0.05.

In [ ]:
import gseapy as gp

from pseudospace.conventional_pt import deseq2_contrast

SCOPES = ['whole_PT', 'S1', 'S2', 'S3']
DE_COLUMNS = ['baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']


def pseudobulk_blocks(ds):
    d = DATA[ds]
    names = d['reference_specimens'] + d['case_specimens']
    blocks = {s: pd.DataFrame(np.vstack([np.asarray(d['counts'][(d['specimen'] == n) & (d['segment'] == s)].sum(axis=0)).ravel()
                                         for n in names]), index=names, columns=d['genes']).round().astype(np.int64)
              for s in ('S1', 'S2', 'S3')}
    blocks['whole_PT'] = sum(blocks[s] for s in ('S1', 'S2', 'S3'))
    return blocks


def de_metadata(ds, label):
    d = DATA[ds]
    names = d['reference_specimens'] + d['case_specimens']
    meta = pd.DataFrame({'condition': ['case' if n in d['case_specimens'] else 'reference' for n in names]}, index=names)
    if label == 'species':
        return meta, '~condition', ('condition', 'case', 'reference')
    meta['group'] = ['a' if n in group_one(ds, label) else 'b' for n in names]
    return meta, '~condition + group', ('group', 'a', 'b')


def de_task(block, meta, design, contrast):
    keep = block.columns[block.sum(axis=0) > 0]
    with threadpool_limits(2), warnings.catch_warnings():
        warnings.simplefilter('ignore')
        result = deseq2_contrast(block[keep], meta, design=design, contrast=contrast)
    return result[['gene', *DE_COLUMNS]]


def signed_gsea(stat, sets):
    ranking = stat.dropna().rename_axis('gene').reset_index(name='stat').sort_values(['stat', 'gene'], ascending=[False, True], kind='stable')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        table = gp.prerank(rnk=ranking, gene_sets=sets, min_size=SET_SIZES[0], max_size=SET_SIZES[1], weight=1,
                           method='multilevel', seed=GSEA_SEED, threads=4, outdir=None, no_plot=True).res2d
    return pd.DataFrame({'pathway_id': table.Term.astype(str).to_numpy(), 'NES': pd.to_numeric(table.NES).to_numpy(),
                         'p': pd.to_numeric(table['NOM p-val']).to_numpy()})


def frame_payload(frame):
    return {c: (frame[c].to_numpy().astype(str) if frame[c].dtype == object else frame[c].to_numpy()) for c in frame.columns}


DE, DE_GSEA, BLOCKS = {}, {}, {}
t0 = time.perf_counter()
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    BLOCKS[ds] = pseudobulk_blocks(ds)
    tasks = [(label, scope) for label in labels for scope in SCOPES]

    def run_de(ds=ds, tasks=tasks):
        with parallel_config(backend='loky', inner_max_num_threads=2):
            parts = Parallel(n_jobs=min(len(tasks), max(1, N_JOBS // 2)))(
                delayed(de_task)(BLOCKS[ds][scope], *de_metadata(ds, label)) for label, scope in tasks)
        return frame_payload(pd.concat([p.assign(partition=label, scope=scope) for (label, scope), p in zip(tasks, parts)],
                                       ignore_index=True))
    got = cached(f'deseq2_{ds}', run_de, params={'pydeseq2': version('pydeseq2'), 'refit_cooks': False, 'scopes': SCOPES},
                 inputs={'blocks': {s: BLOCKS[ds][s].to_numpy() for s in SCOPES}, 'genes': d['genes'],
                         'partitions': {l: group_one(ds, l) for l in labels}})
    DE[ds] = pd.DataFrame(got)

    def run_gsea(ds=ds, tasks=tasks):
        stats = {(l, s): DE[ds][DE[ds].partition.eq(l) & DE[ds].scope.eq(s)].set_index('gene').stat for l, s in tasks}
        with parallel_config(backend='loky', inner_max_num_threads=4):
            parts = Parallel(n_jobs=min(len(tasks), max(1, N_JOBS // 4)))(delayed(signed_gsea)(stats[t], DATA[ds]['sets']) for t in tasks)
        return frame_payload(pd.concat([p.assign(partition=l, scope=s) for (l, s), p in zip(tasks, parts)], ignore_index=True))
    got = cached(f'signed_gsea_{ds}', run_gsea, params={'weight': 1, 'method': 'multilevel', 'seed': GSEA_SEED, 'gseapy': gp.__version__},
                 inputs={'de': DE[ds][['gene', 'stat', 'partition', 'scope']].astype(str).to_numpy(), 'sets': d['sets']})
    g = pd.DataFrame(got)
    g['family'] = np.where(g.scope.eq('whole_PT'), 'whole_PT', 'segments')
    g['q'] = g.groupby(['partition', 'family']).p.transform(lambda p: false_discovery_control(p.fillna(1.).to_numpy()))
    DE_GSEA[ds] = g
print(f'3.4 DESeq2 and signed GSEA: {time.perf_counter() - t0:.0f} s')
DE_GENES = pd.DataFrame([{'dataset': DATA[ds]['title'], 'partition': SHORT[ds][l], 'scope': s,
                          'genes tested': int((DE[ds].partition.eq(l) & DE[ds].scope.eq(s)).sum()),
                          'genes without counts (untested)': len(DATA[ds]['genes']) - int((DE[ds].partition.eq(l) & DE[ds].scope.eq(s)).sum()),
                          'padj ≤ 0.05': int((DE[ds].partition.eq(l) & DE[ds].scope.eq(s) & DE[ds].padj.le(ALPHA)).sum()),
                          'padj ≤ 0.05 and |log2FC| ≥ 1': int((DE[ds].partition.eq(l) & DE[ds].scope.eq(s) & DE[ds].padj.le(ALPHA)
                                                              & DE[ds].log2FoldChange.abs().ge(LFC)).sum())}
                         for ds in DATASETS for l in PARTITIONS[ds] for s in SCOPES])
DE_GENES.to_csv(tables / 'deseq2_gene_counts.csv', index=False)
display(DE_GENES.pivot_table(index=['dataset', 'scope'], columns='partition', values='padj ≤ 0.05', sort=False))


def de_gsea_calls(ds, label, scopes, frame=None):
    """Signed GSEA calls and calibration p of one partition (whole PT or S1/S2/S3); ``frame``: another GSEA table."""
    g = DE_GSEA[ds] if frame is None else frame
    rows = g[g.partition.eq(label) & g.scope.isin(scopes)].dropna(subset=['p'])
    best = rows.groupby('pathway_id').agg(q=('q', 'min'), p=('p', 'min'), tests=('p', 'size'))
    calibration = np.minimum(1., best.p * best.tests) if len(scopes) > 1 else best.p
    return set(best.index[best.q.le(ALPHA)]), calibration, best


def de_ora(ds, label, scopes, keep=None, sets=None):
    """56's ORA on DESeq2 lists of one partition: one family over set x scope x direction. ``keep``: a gene subset whose
    padj is recomputed within it (section 4); ``sets``: defaults to 71's sets."""
    d = DATA[ds]
    sets = d['sets'] if sets is None else sets
    selections, matrices, universes = [], [], []
    for scope in scopes:
        rows = DE[ds][DE[ds].partition.eq(label) & DE[ds].scope.eq(scope)].dropna(subset=['stat']).set_index('gene')
        if keep is not None:
            rows = rows[rows.index.isin(sorted(keep))].copy()
            rows['padj'] = false_discovery_control(rows.pvalue.fillna(1.).to_numpy())
        universe = list(rows.index)
        universes.append(set(universe))
        member = membership(sets, universe)
        for direction in (1, -1):
            selections.append((rows.padj.le(ALPHA) & (direction * rows.log2FoldChange).ge(LFC)).to_numpy())
            matrices.append(member)
    out = family_ora(selections, matrices).set_index(pd.Index(list(sets), name='pathway_id'))
    return out, universes


DE_ROWS = {'deseq2_wholept_gsea': ('whole-PT DESeq2 + signed GSEA', 'average-level'),
           'deseq2_segments_gsea': ('S1/S2/S3 DESeq2 + signed GSEA', 'discrete'),
           'deseq2_wholept_ora': ('whole-PT DESeq2 + ORA', 'average-level'),
           'deseq2_segments_ora': ('S1/S2/S3 DESeq2 + ORA', 'discrete')}
DE_TESTS = {ds: {k: {} for k in DE_ROWS} for ds in DATASETS}
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    for key, scopes in (('deseq2_wholept_gsea', ['whole_PT']), ('deseq2_segments_gsea', ['S1', 'S2', 'S3'])):
        per = {l: de_gsea_calls(ds, l, scopes) for l in labels}
        DE_TESTS[ds][key] = {l: per[l][2].assign(call=per[l][2].q.le(ALPHA)) for l in labels}
        register(ds, key, DE_ROWS[key][0], DE_ROWS[key][1], '75 (31, 56)', 'DESeq2 pseudobulk',
                 {l: per[l][0] for l in labels}, {l: per[l][1].to_numpy(float) for l in labels[1:]}, len(per[labels[0]][2]),
                 'per set: NES and sign, p (per segment)')
    for key, scopes in (('deseq2_wholept_ora', ['whole_PT']), ('deseq2_segments_ora', ['S1', 'S2', 'S3'])):
        per = {l: de_ora(ds, l, scopes) for l in labels}
        DE_TESTS[ds][key] = {l: per[l][0] for l in labels}
        union = set().union(*per[labels[0]][1])
        register(ds, key, DE_ROWS[key][0], DE_ROWS[key][1], '75 (56)', 'DESeq2 pseudobulk',
                 {l: set(per[l][0].index[per[l][0].call]) for l in labels},
                 {l: per[l][0].calibration_p.to_numpy(float) for l in labels[1:]}, len(testable(d['sets'], union)),
                 'per set: enriched DE list (scope, direction)')
    for key in DE_ROWS:
        pd.concat({SHORT[ds][l]: DE_TESTS[ds][key][l] for l in labels}, axis=1).to_csv(tables / f'pathway_tests_{key}_{ds}.csv')
    DE[ds].to_csv(tables / f'deseq2_genes_{ds}.csv', index=False)

## 3.5 · Gene-first curve modules (notebooks 56(c) and 61's B2, on notebook 71's NB Δ(x))

- **Curves:** each gene's Δ(x) of that partition (71's gene model, natural log, 101-point grid), centred and scaled.
  Genes whose fit did not converge or is separated have no usable Δ(x) and are left out of every list.
- **k:** k-means (20 starts, seed 56), k in 3–8 chosen on the condition split by the mean adjusted Rand index over
  20 resamples of 80% of the genes (notebook 56's rule; ties to the smaller k). The relabelings recluster their own
  lists with that k.
- **ORA:** per module against all of 71's genes, BH over module × set (`family_ora`); a call is any module with q ≤
  0.05; the calibration p is the best module p × k.
- **56(c) twin (`modules_top10`):** the top 10% of genes by that partition's z (the primary gene statistic).
- **B2 twin (`modules_significant`):** the genes with q ≤ 0.05 on that partition's z. A list shorter than 10 k genes has
  no modules and no calls (notebook 61's rule).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score


def standardize(x):
    return (x - x.mean(axis=1, keepdims=True)) / np.maximum(x.std(axis=1, keepdims=True), 1e-12)


def kmeans_labels(data, k, seed, pick):
    with threadpool_limits(1):
        rows = data if pick is None else data[pick]
        return KMeans(n_clusters=k, n_init=KM_STARTS, random_state=seed).fit(rows).labels_


def choose_k(stage, data):
    """Notebook 56's rule: the mean ARI of 20 resamples of 80% per k; the most stable k, ties to the smaller."""
    rng = np.random.default_rng(KM_SEED)
    tasks = []
    for k in K_RANGE:
        tasks.append((k, 'full', KM_SEED, None))
        for r in range(N_RESAMPLE):
            tasks.append((k, r, KM_SEED + r + 1, np.sort(rng.choice(len(data), int(.8 * len(data)), replace=False))))

    def compute():
        with parallel_config(backend='loky', inner_max_num_threads=1):
            fitted = Parallel(n_jobs=N_JOBS)(delayed(kmeans_labels)(data, k, seed, pick) for k, _, seed, pick in tasks)
        return {f'{k}|{r}': lab for (k, r, _, _), lab in zip(tasks, fitted)}
    runs = cached(stage, compute, params={'k': list(K_RANGE), 'resamples': N_RESAMPLE, 'seed': KM_SEED, 'starts': KM_STARTS},
                  inputs={'data': digest(data)})
    picks = {f'{k}|{r}': pick for k, r, _, pick in tasks}
    stability = {k: float(np.mean([adjusted_rand_score(runs[f'{k}|full'][picks[f'{k}|{r}']], runs[f'{k}|{r}'])
                                    for r in range(N_RESAMPLE)])) for k in K_RANGE}
    return max(stability, key=lambda k: (round(stability[k], 6), -k)), stability, runs


MODULE_ROWS = {'modules_top10': ('curve modules of the top 10% genes by z + ORA (56(c) twin)', 0),
               'modules_significant': ('curve modules of the q ≤ 0.05 genes + ORA (61 B2 twin)', MIN_PER_MODULE)}
MODULES, MODULE_TABLES, MODULE_K = {ds: {} for ds in DATASETS}, [], []
t0 = time.perf_counter()
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    genes = np.array(d['genes'])
    member = membership(d['sets'], d['genes'])
    n_top = int(round(TOP_SHARE * len(genes)))
    for key, (label_text, min_per_module) in MODULE_ROWS.items():
        lists = {}
        for label in labels:
            frame, delta = STAT71[ds][label], MODELS[ds][label].delta
            eligible = (frame.converged & ~frame.separated).to_numpy() & np.isfinite(delta).all(axis=1)
            if key == 'modules_top10':
                z = np.where(eligible, frame.z.to_numpy(), -np.inf)
                rows = np.sort(np.argsort(-z, kind='stable')[:n_top])
            else:
                rows = np.flatnonzero(eligible & frame.q.le(ALPHA).to_numpy())
            lists[label] = (rows, standardize(delta[rows]))
        k, stability, runs = choose_k(f'kselect_{key}_{ds}', lists[labels[0]][1])
        MODULE_K.append({'dataset': d['title'], 'row': key, 'k': k, **{f'ARI k={kk}': v for kk, v in stability.items()},
                         **{f'genes · {SHORT[ds][l]}': len(lists[l][0]) for l in labels}})
        calls, calibration, per = {}, {}, {}
        for label in labels:
            rows, data = lists[label]
            if len(rows) < max(min_per_module * k, k + 1):
                calls[label], calibration[label] = set(), np.ones(len(d['sets']))
                per[label] = None
                continue
            assign = runs[f'{k}|full'] if label == labels[0] else cached(
                f'modules_{key}_{ds}_{slug(label)}', lambda data=data, k=k: {'labels': kmeans_labels(data, k, KM_SEED, None)},
                params={'k': k, 'seed': KM_SEED, 'starts': KM_STARTS}, inputs={'data': digest(data)})['labels']
            ora = family_ora([np.isin(np.arange(len(genes)), rows[assign == m]) for m in range(k)], member)
            ora.index = pd.Index(list(d['sets']), name='pathway_id')
            calls[label] = set(ora.index[ora.call])
            calibration[label] = ora.calibration_p.to_numpy(float)
            per[label] = ora
            MODULE_TABLES.append(pd.DataFrame({'dataset': ds, 'row': key, 'partition': SHORT[ds][label],
                                               'gene': genes[rows], 'module': assign}))
        MODULES[ds][key] = {'k': k, 'lists': lists, 'ora': per,
                            'labels': {l: (runs[f'{k}|full'] if l == labels[0] else None) for l in labels}}
        register(ds, key, label_text, 'gene-first', '75 (56, 61)', 'NB Δ(x) (71)', calls,
                 {l: calibration[l] for l in labels[1:]}, len(d['sets']), 'modules: genes with one difference shape, their enriched sets')
MODULE_K = pd.DataFrame(MODULE_K)
MODULE_K.to_csv(tables / 'modules_k_selection.csv', index=False)
MODULE_GENES = pd.concat(MODULE_TABLES, ignore_index=True)
for ds in DATASETS:
    MODULE_GENES[MODULE_GENES.dataset.eq(ds)].drop(columns='dataset').to_csv(tables / f'modules_{ds}.csv', index=False)
    for key in MODULE_ROWS:
        per = MODULES[ds][key]['ora']
        pd.concat({SHORT[ds][l]: per[l] for l in per if per[l] is not None}, axis=1).to_csv(tables / f'pathway_tests_{key}_{ds}.csv')
display(MODULE_K.round(3))
print(f'3.5: {time.perf_counter() - t0:.0f} s')

## 3.6 · Rows copied from notebook 71 (no recomputation)

The positional F-test (primary); the clustering method with the default and the single-split null (relabelings under
the fair null); the S1 shape GSEA and the level and total cameraPR and GSEA arms; the S2 ssGSEA level and shape tests;
and the S3 level gene list + ORA (human vs mouse only). Their calls and calibration p are read from 71's tables and its
cached clustering runs, and judged by section 2's rule.

**Guard G4 (not cached):** every copied row reproduces 71's own calls, relabeled calls and verdicts.

In [ ]:
PATHWAY_PARAMS = PathwayParams(min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
EXPRESSION_COLUMN = 'expression_per_10k'


def unflatten(payload):
    names = sorted({k.split('|', 1)[0] for k in payload if not k.startswith('__')})
    return {n: pd.DataFrame({c: payload[f'{n}|{c}'] for c in payload[f'{n}|__columns'].astype(str)}) for n in names}


def b_calls(t):
    """The clustering method's calls of one run: matched ORA q < 0.05 in a gene group or a cluster."""
    return set(t['ora'][t['ora'].q_matched.astype(float) < ALPHA].pathway_id)


def b_bonferroni_p(ds, t):
    """Notebook 61/66's list-based convention: the best matched ORA p over the lists x the number of lists."""
    genes = t['genes']
    n_lists = sum(int((genes[c][genes[c].ne('')].value_counts() >= PATHWAY_PARAMS.min_group_size).sum()) for c in ('group', 'cluster'))
    best = t['ora'].groupby('pathway_id').p_matched.min().astype(float)
    return (best * max(n_lists, 1)).clip(upper=1.).reindex(list(DATA[ds]['sets'])).fillna(1.).to_numpy()


PRIMARY, B, S1, S2, S3 = {}, {ds: {} for ds in DATASETS}, {}, {}, {}
S1_ROWS = {('shape', 'GSEA prerank'): ('s1_shape_gsea', 'positional F-test z · GSEA prerank (71 S1)', 'positional'),
           ('level', 'cameraPR'): ('s1_level_camerapr', 'level z (signed) · cameraPR (71 S1)', 'average-level'),
           ('level', 'GSEA prerank'): ('s1_level_gsea', 'level z (signed) · GSEA prerank (71 S1)', 'average-level'),
           ('total', 'cameraPR'): ('s1_total_camerapr', 'total z · cameraPR (71 S1)', 'average-level'),
           ('total', 'GSEA prerank'): ('s1_total_gsea', 'total z · GSEA prerank (71 S1)', 'average-level')}
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    condition, r1, r2 = labels
    PRIMARY[ds] = {l: pd.read_csv(T71 / 'tables' / f'pathway_tests_{ds}_cameraPR_{slug(l)}.csv').set_index('pathway_id') for l in labels}
    register(ds, 'primary', 'positional F-test · cameraPR on z (primary)', 'positional', '71', 'NB, spline',
             {l: set(PRIMARY[ds][l].index[PRIMARY[ds][l].call.astype(bool)]) for l in labels},
             {l: PRIMARY[ds][l].p_one_sided.to_numpy(float) for l in labels[1:]}, len(d['sets']),
             'per set: p, FDR; per gene: F, z, q, Δ(x)')
    for contrast, null in [(condition, None), (condition, r1), (condition, r2), (r1, r2), (r2, r1)]:
        stage = f'method_b_{ds}_{slug(contrast)}__null_{"default" if null is None else slug(null)}'
        B[ds][(contrast, null)] = unflatten(stored(T71, stage))
    fair = {r1: B[ds][(r1, r2)], r2: B[ds][(r2, r1)]}
    fair_calls, fair_p = {l: b_calls(t) for l, t in fair.items()}, {l: b_bonferroni_p(ds, t) for l, t in fair.items()}
    register(ds, 'clustering', 'clustering · ORA, default null (71)', 'gene-first', '71', 'NB curves (package stages 06–07)',
             {condition: b_calls(B[ds][(condition, None)]), **fair_calls}, fair_p, len(d['sets']),
             'gene groups and clusters with their enriched sets')
    register(ds, 'clustering_single', 'clustering · ORA, single-split null (71)', 'gene-first', '71',
             'NB curves (package stages 06–07)', {condition: set(), **fair_calls}, fair_p, len(d['sets']),
             'as above', condition_runs=[b_calls(B[ds][(condition, r)]) for r in (r1, r2)])
    s1 = pd.read_csv(T71 / 'tables' / f's1_pathway_tests_{ds}.csv', header=[0, 1], index_col=0)
    S1[ds] = s1
    for (arm, test), (key, label_text, family) in S1_ROWS.items():
        per = {l: s1[f'{arm} · {test} · {SHORT[ds][l]}'].dropna(subset=['p']) for l in labels}
        register(ds, key, label_text, family, '71 (S1)', 'NB, spline', {l: set(per[l].index[per[l].hit.astype(bool)]) for l in labels},
                 {l: per[l].p.to_numpy(float) for l in labels[1:]}, len(per[labels[0]]), 'per set: p, FDR (and sign)')
    s2 = pd.read_csv(T71 / 'tables' / f's2_ssgsea_tests_{ds}.csv', header=[0, 1], index_col=0)
    S2[ds] = s2
    for test in ('level', 'shape'):
        per = {l: s2[f'{test} · {SHORT[ds][l]}'] for l in labels}
        register(ds, f's2_{test}', f'ssGSEA + limma, {test} (71 S2)', 'score', '71 (S2)', 'ssGSEA scores (specimen × 6 bins)',
                 {l: set(per[l].index[per[l].q.le(ALPHA)]) for l in labels}, {l: per[l].p.to_numpy(float) for l in labels[1:]},
                 len(s2), 'per set: activity score test')
    if ds == hvm:
        files = {condition: 's3_level_ora_species.csv'}
        for l in (r1, r2):
            mouse = [s for s in group_one(ds, l) if s.startswith('Ctrl')][0]
            files[l] = f's3_level_ora_relabeled_COR1_{mouse}.csv'
        per = {}
        for l, f in files.items():
            ora = pd.read_csv(T71 / 'tables' / f)
            per[l] = ora.groupby('pathway_id').agg(fdr=('fdr', 'min'), p=('p', 'min'), tests=('p', 'size'))
        S3[ds] = per
        register(ds, 's3_level_ora', 'level gene list + ORA (71 S3)', 'average-level', '71 (S3)', 'NB level L (68)',
                 {l: set(per[l].index[per[l].fdr.le(ALPHA)]) for l in labels},
                 {l: np.minimum(1., per[l].p * per[l].tests).to_numpy(float) for l in labels[1:]}, len(per[condition]),
                 'per set: enriched level list (direction)')

# G4: the copied rows reproduce notebook 71's own tables
ROW_TABLE = pd.DataFrame(ROWS)
M71 = pd.read_csv(T71 / 'tables' / 'method_table.csv')
S1P = pd.read_csv(T71 / 'tables' / 's1_pass_table.csv')
S2P = pd.read_csv(T71 / 'tables' / 's2_ssgsea_pass_table.csv')
S3P = pd.read_csv(T71 / 'tables' / 's3_level_variants_summary.csv', index_col=0)
for ds, d in DATA.items():
    mine = ROW_TABLE[ROW_TABLE.dataset.eq(d['title'])].set_index('row')
    for key, method in (('primary', 'normalized · cameraPR on z'), ('clustering', 'clustering · ORA, default null'),
                        ('clustering_single', 'clustering · ORA, single-split null')):
        theirs = M71[M71.dataset.eq(d['title']) & M71.method.eq(method)].iloc[0]
        assert (mine.loc[key, 'calls'], mine.loc[key, 'relabeled calls']) == (str(theirs['calls']), str(theirs['relabeled calls'])), (ds, key)
        assert mine.loc[key, 'relabeling rule'] == theirs['relabeling rule'] and mine.loc[key, 'calibration'] == theirs['calibration'].split(' ')[0]
        assert np.isclose(mine.loc[key, 'calibration share'], theirs['calibration share'])
    for (arm, test), (key, _, _) in S1_ROWS.items():
        theirs = S1P[S1P.dataset.eq(d['title']) & S1P.arm.eq(arm) & S1P.test.eq(test)].iloc[0]
        assert (mine.loc[key, 'calls'], mine.loc[key, 'relabeled calls'], mine.loc[key, 'pass']) == \
            (str(theirs['calls']), theirs['relabeled'], theirs['pass']), (ds, key)
    for test in ('level', 'shape'):
        theirs = S2P[S2P.dataset.eq(d['title']) & S2P['ssGSEA test'].eq(test)].iloc[0]
        assert (mine.loc[f's2_{test}', 'calls'], mine.loc[f's2_{test}', 'relabeled calls'], mine.loc[f's2_{test}', 'pass']) == \
            (str(theirs['calls']), theirs['relabeled'], theirs['pass']), (ds, test)
    if ds == hvm:
        assert int(mine.loc['s3_level_ora', 'calls']) == int(S3P.iloc[0]['calls'])
        assert mine.loc['s3_level_ora', 'relabeled calls'] == f"{int(S3P.iloc[1]['calls'])}, {int(S3P.iloc[2]['calls'])}"
guard_log.append("G4: the rows copied from notebook 71 reproduce its calls, relabeled calls and verdicts")
print(guard_log[-1])
display(ROW_TABLE.set_index(['dataset', 'method']).drop(columns=['returns', 'row']).round(3))

## 4 · Probe exposure (human vs mouse, every row)

The two species' probe panels differ, so a call can rest on genes whose probe counts differ between panels. Each row
is rerun on the **equal-probe genes** (`probe_balance == 'balanced'` in notebook 38's `probe_counts.csv`, the same set
as notebook 56's equal counts), for the condition split, without refitting any gene model:
- set tests on the restricted statistic (rank-normal scores and GSEA rankings recomputed within the restricted genes;
  the joint test refitted on all three partitions, because its VIF comes from the relabelings);
- DE lists with BH recomputed within the restricted genes (56's rule); DE GSEA on the restricted Wald stat;
- modules: the same modules, their ORA with the restricted genes as background;
- clustering: its ORA (`ora_by_group`) on 71's gene groups and clusters with the universe restricted (as 71's
  proteasome sensitivity);
- ssGSEA rescored on the restricted genes; the level gene list with BH within the restricted genes;
- sets re-filtered to 10–300 members.

**Exposure** = 1 − |calls ∩ equal-probe calls| / calls; a row is probe-exposed when exposure ≥ 0.5 (56's rule).

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from scipy.stats import hypergeom
from scipy.stats import t as t_dist

limma = importr('limma')
probes = pd.read_csv(nb38 / 'probe_counts.csv').drop_duplicates('mouse_symbol').set_index('mouse_symbol')
d = DATA[hvm]
condition = list(PARTITIONS[hvm])[0]
GENES_H = pd.Index(d['genes'])
EQUAL_MASK = probes.probe_balance.reindex(GENES_H).eq('balanced').to_numpy()
EQUAL = list(GENES_H[EQUAL_MASK])
SETS_E = restrict_sets(d['sets'], EQUAL)
print(f'Equal-probe genes: {len(EQUAL):,} of {len(GENES_H):,}; sets with 10–300 of them: {len(SETS_E):,} of {len(d["sets"]):,}')


def camera_calls(statistic, sets, *, use_ranks=False, two_sided=False):
    t = camera_pr(statistic, sets, inter_gene_cor=CAMERA_COR, use_ranks=use_ranks, fdr=ALPHA)
    return set(t.pathway_id[t.fdr.le(ALPHA)] if two_sided else t.pathway_id[t.call])


def gsea_calls(statistic, sets, stage, *, metric='raw', two_sided=False):
    g = gsea_frame(cached(stage, lambda: gsea_payload(statistic, sets, metric),
                          params={'permutation_num': GSEA_PERMUTATIONS, 'seed': GSEA_SEED, 'sizes': SET_SIZES, 'metric': metric},
                          inputs={'statistic': statistic.to_numpy(), 'genes': list(statistic.index), 'sets': sets}))
    return set(g.index[g.q.le(ALPHA)] if two_sided else g.index[g.call.astype(bool)])


EQ = {}
z71 = STAT71[hvm][condition].z
EQ['primary'] = [camera_calls(z71[EQUAL_MASK], SETS_E)]
t_spatial = MODELS[hvm][condition].statistics.T_spatial[EQUAL_MASK]
EQ['raw_camerapr'] = [camera_calls(rank_normal_scores(t_spatial), SETS_E)]
EQ['raw_camerapr_ranks'] = [camera_calls(rank_normal_scores(t_spatial), SETS_E, use_ranks=True)]
EQ['raw_gsea_ranknormal'] = [gsea_calls(t_spatial, SETS_E, 'probe_raw_gsea_rank_normal', metric='rank_normal')]
EQ['raw_gsea_raw'] = [gsea_calls(t_spatial, SETS_E, 'probe_raw_gsea_raw', metric='raw')]
EQ['segment'] = [camera_calls(SEG[hvm][condition].z[EQUAL_MASK], SETS_E)]
for key, statistic, frames in (('nb_joint', 'T_spatial', {l: MODELS[hvm][l].statistics.T_spatial.to_numpy() for l in PARTITIONS[hvm]}),
                               ('segment_joint', 'T_step', {l: SEG[hvm][l].T_step.to_numpy() for l in PARTITIONS[hvm]})):
    res, _ = joint_test(hvm, f'matched_{key}_{hvm}_equal_probe', statistic, frames, keep=EQUAL)
    EQ[key] = [set(res[condition].index[res[condition].call.astype(bool)])]

# DE rows
restricted = []
for scope in SCOPES:
    stat = DE[hvm][DE[hvm].partition.eq(condition) & DE[hvm].scope.eq(scope)].set_index('gene').stat
    restricted.append((scope, stat[stat.index.isin(EQUAL)]))
got = cached('probe_signed_gsea', lambda: frame_payload(pd.concat([signed_gsea(s, SETS_E).assign(scope=scope) for scope, s in restricted],
                                                                  ignore_index=True)),
             params={'weight': 1, 'method': 'multilevel', 'seed': GSEA_SEED, 'gseapy': gp.__version__},
             inputs={'stats': {scope: s.to_numpy() for scope, s in restricted}, 'genes': {scope: list(s.index) for scope, s in restricted},
                     'sets': SETS_E})
g_eq = pd.DataFrame(got).assign(partition=condition)
g_eq['family'] = np.where(g_eq.scope.eq('whole_PT'), 'whole_PT', 'segments')
g_eq['q'] = g_eq.groupby('family').p.transform(lambda p: false_discovery_control(p.fillna(1.).to_numpy()))
EQ['deseq2_wholept_gsea'] = [de_gsea_calls(hvm, condition, ['whole_PT'], frame=g_eq)[0]]
EQ['deseq2_segments_gsea'] = [de_gsea_calls(hvm, condition, ['S1', 'S2', 'S3'], frame=g_eq)[0]]
for key, scopes in (('deseq2_wholept_ora', ['whole_PT']), ('deseq2_segments_ora', ['S1', 'S2', 'S3'])):
    out, _ = de_ora(hvm, condition, scopes, keep=EQUAL, sets=SETS_E)
    EQ[key] = [set(out.index[out.call])]

# modules: the same modules, background the equal-probe genes
member_e = membership(SETS_E, EQUAL)
for key in MODULE_ROWS:
    m = MODULES[hvm][key]
    rows, assign = m['lists'][condition][0], m['labels'][condition]
    module_genes = [set(GENES_H[rows[assign == j]]) for j in range(m['k'])]
    ora = family_ora([np.isin(EQUAL, sorted(g)) for g in module_genes], member_e)
    EQ[key] = [set(np.array(list(SETS_E))[ora.call.to_numpy()])]

# clustering: ORA on 71's groups and clusters with the universe restricted
ORA_KW = dict(n_strata=PATHWAY_PARAMS.n_strata, min_set_size=PATHWAY_PARAMS.min_genes,
              min_group_size=PATHWAY_PARAMS.min_group_size, min_overlap=PATHWAY_PARAMS.min_overlap)


def clustering_calls(genes_table, sets, universe=None):
    expression = genes_table[EXPRESSION_COLUMN].to_numpy(float)
    ora = pd.concat([ora_by_group(genes_table[k].to_numpy(), sets, expression, universe=universe, **ORA_KW).assign(list=k)
                     for k in ('group', 'cluster')], ignore_index=True)
    return set((ora.library + '::' + ora.pathway)[ora.q_matched < ALPHA])


r1, r2 = list(PARTITIONS[hvm])[1:]
for key, runs in (('clustering', [(condition, None)]), ('clustering_single', [(condition, r1), (condition, r2)])):
    EQ[key] = []
    for run in runs:
        table = B[hvm][run]['genes']
        assert list(table.gene.astype(str)) == d['genes']
        assert clustering_calls(table, EXT_SETS[hvm]) == b_calls(B[hvm][run]), 'the clustering ORA does not reproduce the run'
        EQ[key].append(clustering_calls(table, EXT_SETS[hvm], universe=EQUAL_MASK))

# S1 arms (71's gene z)
s1z = pd.read_csv(T71 / 'tables' / f's1_gene_z_{hvm}.csv', header=[0, 1], index_col=0)
arm_column = {'shape': 'shape (T_spatial)', 'level': 'level signed (T_level)', 'total': 'total (T_total)'}
for (arm, test), (key, _, _) in S1_ROWS.items():
    z = s1z[('condition', arm_column[arm])].reindex(GENES_H)[EQUAL_MASK]
    if test == 'cameraPR':
        EQ[key] = [camera_calls(z, SETS_E, two_sided=arm == 'level')]
    else:
        EQ[key] = [gsea_calls(z, SETS_E, f'probe_s1_gsea_{arm}', two_sided=arm == 'level')]


# S2: ssGSEA rescored on the equal-probe genes (71's S2 code, condition split)
def limma_test(Y, X, coefs):
    """Y sets x samples, X design (samples x p); moderated F (or t) p-values for coefs (0-based)."""
    rY = ro.r.matrix(ro.FloatVector(Y.to_numpy().ravel()), nrow=Y.shape[0], byrow=True)
    rX = ro.r.matrix(ro.FloatVector(X.T.ravel()), nrow=X.shape[0])
    fit = limma.eBayes(limma.lmFit(rY, rX))
    tt = limma.topTable(fit, coef=ro.IntVector([c + 1 for c in coefs]), number=Y.shape[0], sort_by='none')
    return pd.DataFrame({'p': np.asarray(tt.rx2('P.Value')), 'q': np.asarray(tt.rx2('adj.P.Val'))}, index=Y.index)


def ssgsea_condition(ds, keep_mask, sets):
    d_ = DATA[ds]
    cuts = np.quantile(d_['position'], np.linspace(0, 1, 7)[1:-1])
    b = np.searchsorted(cuts, d_['position'], side='right')
    counts = d_['counts'][:, np.flatnonzero(keep_mask)]
    cols = {}
    for n in np.unique(d_['specimen']):
        for k in range(6):
            m = (d_['specimen'] == n) & (b == k)
            cols[f'{n}|{k}'] = np.log2(np.asarray(counts[m].sum(axis=0)).ravel() / d_['library'][m].sum() * 1e6 + 1)
    E = pd.DataFrame(cols, index=np.array(d_['genes'])[keep_mask])
    res = gp.ssgsea(data=E, gene_sets=sets, outdir=None, sample_norm_method='rank', min_size=SET_SIZES[0],
                    max_size=SET_SIZES[1], no_plot=True, threads=N_JOBS, verbose=False)
    S = res.res2d.pivot(index='Term', columns='Name', values='NES').astype(float)[list(E.columns)]
    return {'scores': S.to_numpy(float), 'sets': S.index.to_numpy().astype(str), 'samples': S.columns.to_numpy().astype(str)}


g = cached('probe_s2_ssgsea', lambda: ssgsea_condition(hvm, EQUAL_MASK, SETS_E),
           params={'bins': 6, 'gseapy': gp.__version__, 'sizes': SET_SIZES, 'norm': 'rank'},
           inputs={**d['input_key'], 'keep': EQUAL, 'sets': SETS_E})
S_eq = pd.DataFrame(g['scores'], index=g['sets'].astype(str), columns=g['samples'].astype(str))
meta = pd.DataFrame([c.split('|') for c in S_eq.columns], columns=['specimen', 'bin'], index=S_eq.columns).astype({'bin': int})
names = sorted(meta.specimen.unique())
gs = pd.Series(PARTITIONS[hvm][condition][0], index=d['specimen']).groupby(level=0).first()
L_eq = S_eq.T.groupby(meta.specimen).mean().T[names]
level_eq = limma_test(L_eq, np.column_stack([np.ones(len(names)), gs[names].to_numpy()]), [1])
sp = np.column_stack([(meta.specimen == n).to_numpy(float) for n in names])
bins6 = np.column_stack([(meta.bin == k).to_numpy(float) for k in range(1, 6)])
shape_eq = limma_test(S_eq, np.column_stack([sp, bins6, meta.specimen.map(gs).to_numpy(float)[:, None] * bins6]),
                      list(range(len(names) + 5, len(names) + 10)))
EQ['s2_level'] = [set(level_eq.index[level_eq.q.le(ALPHA)])]
EQ['s2_shape'] = [set(shape_eq.index[shape_eq.q.le(ALPHA)])]

# S3: 71's species t of the level, BH within the equal-probe genes
g3 = pd.read_csv(T71 / 'tables' / 's3_level_gene_table.csv', index_col=0)
logmean3 = pd.Series(d['log_mean_count'], index=GENES_H).reindex(g3.index)
sv = limma.squeezeVar(ro.FloatVector(((g3.dh ** 2 + g3.dm ** 2) / 4).to_numpy()), 2, covariate=ro.FloatVector(logmean3.to_numpy()))
t3 = g3.L.to_numpy() / np.sqrt(np.asarray(sv.rx2('var.post')))
p3 = 2 * t_dist.sf(np.abs(t3), 2 + np.asarray(sv.rx2('df.prior')))


def level_list_ora(keep_mask):
    genes3 = g3.index[keep_mask]
    p, t = p3[keep_mask], t3[keep_mask]
    q = false_discovery_control(p)
    idx = {pid: [genes3.get_loc(x) for x in sorted(set(v) & set(genes3))] for pid, v in d['sets'].items()}
    idx = {pid: v for pid, v in idx.items() if len(v) >= 2}
    rows_ = []
    for direction, sel in (('Up', (q <= ALPHA) & (t > 0)), ('Down', (q <= ALPHA) & (t < 0))):
        K = int(sel.sum())
        for pid, v in idx.items():
            x = int(sel[v].sum())
            rows_.append((pid, direction, hypergeom.sf(x - 1, len(genes3), K, len(v)) if K else 1.))
    ora = pd.DataFrame(rows_, columns=['pathway_id', 'direction', 'p'])
    ora['fdr'] = false_discovery_control(ora.p.to_numpy())
    return set(ora.pathway_id[ora.fdr.le(ALPHA)])


assert level_list_ora(np.ones(len(g3), bool)) == CALLS[hvm]['s3_level_ora']['condition'][0], "S3 does not reproduce notebook 71"
EQ['s3_level_ora'] = [level_list_ora(g3.index.isin(EQUAL))]

PROBE_ROWS = []
for key, eq_runs in EQ.items():
    runs = CALLS[hvm][key]['condition']
    called = sum(len(r) for r in runs)
    kept = sum(len(r & e) for r, e in zip(runs, eq_runs))
    untestable = sum(len(r - set(SETS_E)) for r in runs)
    exposure = 1 - kept / called if called else np.nan
    PROBE_ROWS.append({'row': key, 'method': ROW_INFO[key]['label'], 'calls': ', '.join(str(len(r)) for r in runs),
                       'equal-probe calls': ', '.join(str(len(e)) for e in eq_runs), 'calls kept': kept,
                       'calls untestable on equal-probe genes': untestable, 'exposure': exposure,
                       'probe-exposed': bool(exposure >= EXPOSED) if called else np.nan})
PROBE = pd.DataFrame(PROBE_ROWS)
assert set(PROBE.row) == {r['row'] for r in ROWS if r['dataset'] == DATA[hvm]['title']}, 'a row has no probe-exposure result'
PROBE.to_csv(tables / 'probe_exposure.csv', index=False)
display(PROBE.set_index('method').drop(columns='row').round(3))

## 5 · Overlap with the primary and the offset lean of the calls

- For the primary against every other row (condition split): both, primary only, other only, neither and the
  Jaccard index, over sets and over notebook 71's redundancy clusters ("neither" counts what no call of either touches).
  The clustering method's single-split row has two condition runs and is represented by its default-null row.
- **Offset lean** of a set, from notebook 71's S1 tables (condition split): the set leans to the offset when its
  level-arm |z| (cameraPR, two-sided p) exceeds its shape-arm z (cameraPR, one-sided p). Reported per quadrant
  (notebook 61: 66 of 92 ORA-only calls leaned to the offset, 0 of the continuous calls).

In [ ]:
OVERLAP_ROWS = []
for ds, d in DATA.items():
    s1 = S1[ds]
    level_p = s1['level · cameraPR · condition'].p_two_sided.astype(float).clip(1e-300, 1)
    shape_p = s1['shape · cameraPR · condition'].p.astype(float).clip(1e-300, 1 - 1e-16)
    lean = pd.Series(norm.isf(level_p / 2) > norm.isf(shape_p), index=s1.index)
    primary = CALLS[ds]['primary']['condition'][0]
    all_sets, all_clusters = set(d['sets']), set(REDUNDANCY[ds].representative)
    for key in [r['row'] for r in ROWS if r['dataset'] == d['title'] and r['row'] not in ('primary', 'clustering_single')]:
        other = CALLS[ds][key]['condition'][0]
        rp, ro_ = representatives(ds, primary), representatives(ds, other)
        OVERLAP_ROWS.append({
            'dataset': d['title'], 'row': key, 'method': ROW_INFO[key]['label'], 'family': ROW_INFO[key]['family'],
            'primary calls': len(primary), 'row calls': len(other), 'both': len(primary & other),
            'primary only': len(primary - other), 'row only': len(other - primary), 'neither': len(all_sets - primary - other),
            'Jaccard (sets)': len(primary & other) / len(primary | other) if primary | other else np.nan,
            'clusters: both': len(rp & ro_), 'clusters: primary only': len(rp - ro_), 'clusters: row only': len(ro_ - rp),
            'clusters: neither': len(all_clusters - rp - ro_),
            'Jaccard (redundancy clusters)': len(rp & ro_) / len(rp | ro_) if rp | ro_ else np.nan,
            'offset-leaning · primary only': float(lean.reindex(sorted(primary - other)).mean()) if primary - other else np.nan,
            'offset-leaning · both': float(lean.reindex(sorted(primary & other)).mean()) if primary & other else np.nan,
            'offset-leaning · row only': float(lean.reindex(sorted(other - primary)).mean()) if other - primary else np.nan})
    OVERLAP_ROWS.append({'dataset': d['title'], 'row': 'all sets', 'method': 'every tested set',
                         'offset-leaning · both': float(lean.mean())})
OVERLAP = pd.DataFrame(OVERLAP_ROWS)
for ds in DATASETS:
    OVERLAP[OVERLAP.dataset.eq(DATA[ds]['title'])].to_csv(tables / f'overlap_{ds}.csv', index=False)
display(OVERLAP.set_index(['dataset', 'method'])[['primary calls', 'row calls', 'both', 'primary only', 'row only',
                                                  'Jaccard (sets)', 'Jaccard (redundancy clusters)', 'offset-leaning · primary only',
                                                  'offset-leaning · both', 'offset-leaning · row only']].round(3))

## 6 · Programs and the Figure 4e heatmaps (pathway-first against gene-first)

Notebook 75 owns the programs and the Figure 4e heatmap (O4); later notebooks read `programs_{ds}.csv`.
- **Programs**, per dataset: `pathway_decoys.overlap_programs` on notebook 71's calls sorted by `pathway_id`
  (notebook 61's order-invariant rule; average linkage on 1 − overlap coefficient, cut 0.5). Contributing genes: the
  top 10% by the condition z; all members when fewer than 3. A program is named after its member with the lowest FDR.
- **Heatmap**, one per dataset: the program with the most calls (ties: lowest best FDR). Rows: its contributing genes,
  at most 40 by z, ordered by reference then case peak bin. Columns: specimen × 12 equal bins of the position over
  notebook 71's grid range, mean log1p CP10k (mt-free library), z-scored within group (top: group means; bottom: each
  specimen). Side bars: z, whole-PT DESeq2 log2FC (3.4) and, in human vs mouse only, notebook 73's symmetric class.
- **Gene-first panels:** the B2-twin module (3.5) with the most enriched sets, and the clustering method's cluster with
  the most calls (71's condition run, default null), same layout with their enriched sets (at most five).

Figure 4e is the human vs mouse set; the AKI set is supplementary.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

from pseudospace.pathway_decoys import overlap_programs

HEAT = LinearSegmentedColormap.from_list('teal_grey_brown', ['#01665E', '#5AB4AC', '#E6E6E6', '#D8B365', '#8C510A'])
CLASS_COLORS = {'conserved': '#009E73', 'reversal': '#CC79A7', 'mouse-only': REFERENCE_COLOR, 'human-only': CASE_COLOR,
                'indeterminate': '#E3E3E3', 'neither': '#B8B8B8'}
CLASSES73 = pd.read_csv(nb73_classes, index_col=0)['union S2-S1 + S3c-early']


def zscore_rows(values):
    centred = values - np.nanmean(values, axis=1, keepdims=True)
    sd = np.nanstd(values, axis=1, keepdims=True)
    return np.where(sd > 1e-9, centred / np.where(sd > 1e-9, sd, 1), 0.)


def specimen_bin_means(ds, gene_names):
    """Mean log1p CP10k (mt-free library) per specimen x 12 equal position bins over 71's grid range."""
    d = DATA[ds]
    grid = MODELS[ds]['species'].grid
    bins = np.clip(((d['position'] - grid[0]) / (grid[-1] - grid[0]) * HEAT_BINS).astype(int), 0, HEAT_BINS - 1)
    idx = pd.Index(d['genes']).get_indexer(gene_names)
    y = np.log1p(d['counts'][:, idx].toarray() / d['library'][:, None] * 1e4)
    out = {}
    for n in d['reference_specimens'] + d['case_specimens']:
        rows = d['specimen'] == n
        out[n] = np.column_stack([y[rows & (bins == b)].mean(axis=0) if (rows & (bins == b)).any() else np.full(len(idx), np.nan)
                                  for b in range(HEAT_BINS)])
    return out


def ordered_rows(ds, gene_names):
    """At most 40 genes by condition z, ordered by reference then case peak bin."""
    z = STAT71[ds]['species'].z
    shown = list(z.reindex(gene_names).nlargest(MAX_ROWS).index) if len(gene_names) > MAX_ROWS else list(gene_names)
    means = specimen_bin_means(ds, shown)
    d = DATA[ds]
    ref = zscore_rows(np.nanmean([means[n] for n in d['reference_specimens']], axis=0))
    case = zscore_rows(np.nanmean([means[n] for n in d['case_specimens']], axis=0))
    order = pd.DataFrame({'r': np.nanargmax(ref, axis=1), 'c': np.nanargmax(case, axis=1), 'g': shown}).sort_values(['r', 'c', 'g']).index
    return [shown[i] for i in order]


HEAT_TABLES = []


def draw_heatmap(ds, panel, rows, member_pathways, title, stem):
    d = DATA[ds]
    means = specimen_bin_means(ds, rows)
    groups = {d['reference_name']: d['reference_specimens'], d['case_name']: d['case_specimens']}
    group_z = {g: zscore_rows(np.nanmean([means[n] for n in names], axis=0)) for g, names in groups.items()}
    specimen_z = {n: zscore_rows(means[n]) for n in means}
    for column, values in [*group_z.items(), *specimen_z.items()]:
        HEAT_TABLES.append(pd.DataFrame(values, index=rows, columns=range(HEAT_BINS)).rename_axis('gene').reset_index()
                           .melt(id_vars='gene', var_name='bin', value_name='z').assign(dataset=ds, panel=panel, column=column))
    n, k = len(rows), len(member_pathways)
    height_mm = 40 + n * 3.6 + 3 * k
    fig = plt.figure(figsize=(180 * MM, height_mm * MM))
    widths = [max(k, 1) * .9, 12, 1.2, 12, 1.2, 2.4, 2.4]
    outer = fig.add_gridspec(2, 1, height_ratios=[1, .6], hspace=.16, top=1 - (14 + 3 * k) / height_mm, bottom=.04, left=.12, right=.98)
    top, bottom = outer[0].subgridspec(1, 7, width_ratios=widths, wspace=.06), outer[1].subgridspec(1, 7, width_ratios=widths, wspace=.06)
    mx = fig.add_subplot(top[0, 0])
    matrix = np.array([[g in set(d['sets'][p]) for p in member_pathways] for g in rows], float).reshape(n, k)
    yy, xx = np.nonzero(matrix)
    mx.scatter(xx, yy, s=4, color=INK, linewidths=0)
    mx.set_xlim(-.5, max(k, 1) - .5)
    mx.set_ylim(n - .5, -.5)
    mx.set_xticks(range(k), [str(i + 1) for i in range(k)], fontsize=5)
    mx.xaxis.tick_top()
    mx.set_yticks(range(n), rows, fontsize=5)
    for side in ('right', 'bottom'):
        mx.spines[side].set_visible(False)
    image = None
    for col, (g, names) in zip((1, 3), groups.items()):
        ax = fig.add_subplot(top[0, col])
        image = ax.imshow(np.clip(group_z[g], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP, aspect='auto', interpolation='nearest')
        ax.scatter(np.nanargmax(group_z[g], axis=1), np.arange(n), s=3, color=INK, zorder=3, linewidths=0)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'{g}: specimen mean, z within {g}', loc='left', color=REFERENCE_COLOR if col == 1 else CASE_COLOR)
        sub = bottom[0, col].subgridspec(1, 2, wspace=.08)
        for j, name in enumerate(names):
            bx = fig.add_subplot(sub[0, j])
            bx.imshow(np.clip(specimen_z[name], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP, aspect='auto', interpolation='nearest')
            bx.set_xticks([0, HEAT_BINS - 1], ['start', 'end'], fontsize=4.5)
            bx.set_yticks([])
            bx.set_title(name, loc='left', fontsize=5.5, fontweight='normal')
    cx = fig.add_subplot(top[0, 4])
    if ds == hvm:
        for i, g in enumerate(rows):
            c = CLASSES73.get(g, np.nan)
            cx.add_patch(plt.Rectangle((0, i - .4), 1, .8, color=CLASS_COLORS.get(c, '#FFFFFF'), ec=LIGHT if c not in CLASS_COLORS else 'none', lw=.3))
        cx.set_title('class', loc='left', fontsize=5.5)
    cx.set_xlim(0, 1)
    cx.set_ylim(n - .5, -.5)
    cx.axis('off')
    tx = fig.add_subplot(top[0, 5])
    tx.barh(np.arange(n), STAT71[ds]['species'].z.reindex(rows).to_numpy(), color=FAMILY_COLOR['positional'], height=.7)
    tx.set_ylim(n - .5, -.5)
    tx.set_yticks([])
    tx.set_title('z', loc='left', fontsize=5.5)
    tx.tick_params(labelsize=5)
    lx = fig.add_subplot(top[0, 6])
    whole = DE[ds][DE[ds].partition.eq('species') & DE[ds].scope.eq('whole_PT')].set_index('gene').log2FoldChange
    lfc = whole.reindex(rows).clip(-6, 6).fillna(0).to_numpy()
    lx.barh(np.arange(n), lfc, color=np.where(lfc > 0, CASE_COLOR, REFERENCE_COLOR), height=.7)
    lx.axvline(0, color=INK, lw=.5)
    lx.set_xlim(-6.5, 6.5)
    lx.set_ylim(n - .5, -.5)
    lx.set_yticks([])
    lx.set_title(f'log2 {d["case_name"][0].upper()}/{d["reference_name"][0].upper()}', loc='left', fontsize=5.5)
    lx.tick_params(labelsize=5)
    cax = fig.add_subplot(bottom[0, 4:])
    cax.axis('off')
    bar = fig.colorbar(image, ax=cax, orientation='horizontal', fraction=.45, aspect=10, pad=0)
    bar.set_label(f'z within group (clipped ±{Z_CLIP})', fontsize=5)
    bar.ax.tick_params(labelsize=5)
    if ds == hvm:
        cax.legend([plt.Rectangle((0, 0), 1, 1, color=v) for v in CLASS_COLORS.values()], list(CLASS_COLORS), loc='upper left',
                   fontsize=4.5, ncol=2, handlelength=1, borderaxespad=0)
    fig.text(.01, 1 - 4 / height_mm, title, ha='left', va='top', fontsize=6.5, fontweight='bold')
    fig.text(.01, 1 - 8 / height_mm, '; '.join(f'{i + 1} {NAMES[ds][p]}' for i, p in enumerate(member_pathways)),
             ha='left', va='top', fontsize=5, wrap=True, color=MUTED)
    save(fig, stem)
    plt.show()


PROGRAM_TABLES, HEAT_CHOICES = [], []
for ds, d in DATA.items():
    z = STAT71[ds]['species'].z
    contributing = set(z.nlargest(int(round(TOP_SHARE * len(z)))).index)
    calls71 = sorted(CALLS[ds]['primary']['condition'][0])
    fdr = PRIMARY[ds]['species'].fdr
    programs = overlap_programs(d['sets'], calls71, contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program
    names_ = {}
    for program, block in programs.groupby(programs):
        best = fdr.reindex(block.index).sort_index().sort_values(kind='stable').index[0]
        names_[program] = NAMES[ds][best]
    table = pd.DataFrame({'program': programs, 'program name': programs.map(names_), 'fdr': fdr.reindex(programs.index),
                          'name': NAMES[ds].reindex(programs.index)})
    sizes = table.groupby('program').agg(n=('fdr', 'size'), best=('fdr', 'min'))
    table['program size'] = table.program.map(sizes.n)
    table.sort_values(['program', 'fdr']).to_csv(tables / f'programs_{ds}.csv')
    PROGRAM_TABLES.append({'dataset': d['title'], 'calls': len(calls71), 'programs': int(programs.nunique())})
    chosen = sizes.sort_values(['n', 'best'], ascending=[False, True]).index[0]
    members = sorted(table.index[table.program.eq(chosen)], key=lambda p: (fdr[p], p))
    union = []
    for p in members:
        chosen_genes = [g for g in d['sets'][p] if g in contributing]
        union.extend(chosen_genes if len(chosen_genes) >= 3 else d['sets'][p])
    rows = ordered_rows(ds, sorted(set(union)))
    draw_heatmap(ds, 'program', rows, members, f'{d["title"]} · pathway-first: program "{names_[chosen]}" '
                 f'({len(members)} of {len(calls71)} positional F-test calls) · {len(rows)} of {len(set(union))} genes',
                 f'fig4e_program_{ds}')
    HEAT_CHOICES.append({'dataset': d['title'], 'panel': 'program', 'choice': names_[chosen], 'sets': len(members),
                         'genes': len(set(union)), 'shown': len(rows)})
    module = MODULES[ds]['modules_significant']
    ora = module['ora']['species']
    if ora is not None:
        enriched = [(ora[f'q_{j}'] <= ALPHA).sum() for j in range(module['k'])]
        j = int(np.argmax(enriched))
        rows_m, assign = module['lists']['species'][0], module['labels']['species']
        genes_m = list(np.array(d['genes'])[rows_m[assign == j]])
        top_sets = list(ora[f'q_{j}'][ora[f'q_{j}'] <= ALPHA].sort_values().index[:5])
        rows = ordered_rows(ds, genes_m)
        draw_heatmap(ds, 'module', rows, top_sets, f'{d["title"]} · gene-first: module {j + 1} of the q ≤ 0.05 genes '
                     f'({len(genes_m)} genes, {enriched[j]} enriched sets) · top {len(rows)} by z', f'fig4e_module_{ds}')
        HEAT_CHOICES.append({'dataset': d['title'], 'panel': 'module', 'choice': f'module {j + 1}', 'sets': int(enriched[j]),
                             'genes': len(genes_m), 'shown': len(rows)})
    run = B[ds][('species', None)]
    ora_b = run['ora'][run['ora'].list.eq('cluster') & run['ora'].q_matched.astype(float).lt(ALPHA)]
    if len(ora_b):
        counts_b = ora_b.groupby('group').pathway_id.nunique().sort_values(ascending=False, kind='stable')
        cluster = counts_b.index[0]
        genes_b = list(run['genes'].gene[run['genes'].cluster.eq(cluster)].astype(str))
        top_sets = list(ora_b[ora_b.group.eq(cluster)].sort_values('q_matched').drop_duplicates('pathway_id').pathway_id[:5])
        rows = ordered_rows(ds, genes_b)
        draw_heatmap(ds, 'cluster', rows, top_sets, f'{d["title"]} · clustering method: cluster {cluster} '
                     f'({len(genes_b)} genes, {int(counts_b.iloc[0])} called sets) · top {len(rows)} by z', f'fig4e_cluster_{ds}')
        HEAT_CHOICES.append({'dataset': d['title'], 'panel': 'clustering cluster', 'choice': str(cluster),
                             'sets': int(counts_b.iloc[0]), 'genes': len(genes_b), 'shown': len(rows)})
HEAT_LONG = pd.concat(HEAT_TABLES, ignore_index=True)
for ds in DATASETS:
    HEAT_LONG[HEAT_LONG.dataset.eq(ds)].drop(columns='dataset').to_csv(tables / f'program_heatmap_{ds}.csv', index=False)
PROGRAM_TABLE, HEAT_CHOICES = pd.DataFrame(PROGRAM_TABLES), pd.DataFrame(HEAT_CHOICES)
display(PROGRAM_TABLE)
display(HEAT_CHOICES)

## 7 · The level positive control on the package (notebook 68, human vs mouse)

Notebook 68's protocol, unchanged except the inputs. **It is a positive control of the measurement, not a discovery
screen: no species-level p-value is computed on our data.**
- **L_g** = (1/ln 2) · mean over the grid of [Δ_g(x) − m(x)], m(x) the median of Δ at x over the reference genes
  (converged, not separated), on notebook 71's condition-split model; non-converged genes are dropped. The centring at
  each x removes depth, library composition and the mt share, and any efficiency shared by a whole panel.
  **Guard:** L equals notebook 71's `s3_level_gene_table.csv` on shared genes.
- **Pair spread**, same-species bands and **P_g** (specimen × segment pseudobulk log-CPM, equal segment weights) as in
  68, on the package units, labels and the mt-free library.
- **A_g**: Lake 7 donors against 6 Census adult males, limma-trend robust, centred by the median, refitted on the
  analysed genes (notebook 71's universe present in both atlases). Sensitivities: all 12 Census males, Lake's men,
  Census human for Lake.
- **Criteria** (fixed by 68): (i) ≥ 80% of the primary literature controls with the expected sign (≥ 10 evaluable);
  (ii) ≥ 80% of atlas-strong genes (BH ≤ 0.05, |A| ≥ 1) with |L| ≥ 1 share A's sign (≥ 30 genes); (iii) Spearman L vs A
  (descriptive); (iv) families in the member-count direction (descriptive).
- **Families:** sums over every panel member of notebook 62's orthogroups, from the package's stage-01 native unit
  matrices (`runs/hvm/01_units/{sample}_units.h5ad`, joined on the unit id), against the gene model's library.

Literature controls and the housekeeping list are read only if byte-identical to the files 68 froze.

In [ ]:
import h5py

from pseudospace.replication_rules import csr_group_sums
from pseudospace.species_level import (band, bootstrap_spearman, clears_band, equal_segment_log_cpm, family_sums,
                                       grid_centred_level, limma_trend_species, pair_levels, pair_summary,
                                       segment_pseudobulk, species_difference, wilson_interval)

LITERATURE_SHA256 = 'f1aa71844552327e7f8b11905167fae34c205398c2499053711bcbee2ca06c4f'
HOUSEKEEPING_SHA256 = 'df7aa10e635efb5b0df7dd607e5c11c91df2bdf3e321fb97c0c088ea96a60756'
EXPECTED_SIGN = {'higher in human': 1, 'higher in mouse': -1}
SEGMENTS = ('S1', 'S2', 'S3')
MIN_NUCLEI = 50
LAKE_DONORS = ['164-10', '164-6', '18-142', '18-312', '3535', 'KRP446', 'KRP460']
ADULT_STAGES = ('3-6 month-old mature stage', '12-month-old stage')
N_ADULT_MALES, N_MALES, N_CENSUS_HUMAN = 6, 12, 6
CENSUS_SEGMENTS = {'epithelial cell of proximal tubule segment 1': 'S1', 'epithelial cell of proximal tubule segment 2': 'S2',
                   'epithelial cell of proximal tubule segment 3': 'S3'}
CENSUS_HUMAN_SEGMENTS = {'kidney proximal convoluted tubule epithelial cell': 'PCT',
                         'epithelial cell of proximal tubule segment 3': 'S3'}
CENSUS_HUMAN_WEIGHTS = (2 / 3, 1 / 3)
ATLAS_FDR, ATLAS_LFC = .05, 1.
LEVEL_MIN, PASS_SHARE = 1., .80
MIN_LITERATURE, MIN_ATLAS_GENES = 10, 30
BAND_QUANTILES = (.025, .975)
N_BOOT, SEED = 2000, 68
ABSENT = .01
FAMILY_CLASSES = ('1:many', 'many:1', 'many:many')
FAMILY_DETECTION = .02
PRENAMED = {'Cyp4a': 'Cyp4a10', 'Akr1c': 'Akr1c6', 'Slco1a': 'Slco1a1', 'Sult2a': 'Sult2a1', 'Cyp2d': 'Cyp2d22',
            'Ces1': 'Ces1d', 'Nat8': 'Nat8f1', 'Cyp2j': 'Cyp2j5', 'Cyp2c': 'Cyp2c29', 'Sult1a': 'Sult1a1', 'Ugt2b': 'Ugt2b5'}
GREY = '#a8a7a2'
AGREE_COLOR, DISAGREE_COLOR = '#2a78d6', '#eb6834'


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


if sha256(LITERATURE) != LITERATURE_SHA256 or sha256(HOUSEKEEPING) != HOUSEKEEPING_SHA256:
    raise ValueError('The literature controls or the housekeeping list differ from the frozen files.')
guard_log.append('level control: literature controls and housekeeping list match their frozen sha256')

t0 = time.perf_counter()
d = DATA[hvm]
GENES = pd.Index(d['genes'], name='gene')
counts, library = d['counts'], d['library']
specimen, segment, human = d['specimen'], d['segment'], d['case']
structure_ids = d['obs'].index.astype(str).to_numpy()
GENE_SETS = d['sets']
MODEL = MODELS[hvm]['species']
MOUSE, HUMAN = d['reference_specimens'], d['case_specimens']
assert MOUSE == ['Ctrl1A2', 'Ctrl1A4'] and HUMAN == ['HUK1_COR1', 'HUK1_MED1']

to_human = dict(zip(orthologs.mouse_symbol, orthologs.human_symbol))
to_mouse = dict(zip(orthologs.human_symbol.str.upper(), orthologs.mouse_symbol))
human_symbol = pd.Series(GENES.map(to_human), index=GENES)
inventory = {sp: pd.read_csv(nb62 / 'tables' / f'gene_inventory_{sp}.csv',
                             usecols=['gene', 'gene_id', 'component', 'ortholog_class', 'probe_clean', 'n_human', 'n_mouse'])
             for sp in ('mouse', 'human')}
dropped_pairs = set(pd.read_csv(nb64 / 'tables' / 'dropped_pairs.csv').mouse_symbol)
housekeeping = set(pd.read_csv(HOUSEKEEPING, sep=r'\s+', header=None, names=['symbol', 'refseq']).symbol.str.upper())
audit = pd.read_csv(CENSUS / 'census_symbol_audit.csv')
census_other = {sp: set(audit.column[audit.species.eq(sp) & audit.status.eq('other_id')]) for sp in ('mouse', 'human')}
detection = {side: np.asarray((counts[mask] > 0).mean(axis=0)).ravel() for side, mask in (('human', human), ('mouse', ~human))}
specimen_totals = np.array([np.asarray(counts[specimen == s].sum(axis=0)).ravel() for s in MOUSE + HUMAN])
clean = {sp: inventory[sp].assign(key=inventory[sp].gene.str.upper()).set_index('key').probe_clean.astype(bool)
         for sp in ('mouse', 'human')}
clean = {sp: c[~c.index.duplicated()] for sp, c in clean.items()}
INFO = pd.DataFrame({
    'human_symbol': human_symbol, 'detection_human': detection['human'], 'detection_mouse': detection['mouse'],
    'nearly_absent': np.minimum(detection['human'], detection['mouse']) < ABSENT,
    'zero_in_a_specimen': (specimen_totals == 0).any(axis=0),
    'mouse_probes': probes.mouse_probes.reindex(GENES).to_numpy(), 'human_probes': probes.human_probes.reindex(GENES).to_numpy(),
    'equal_probes': probes.probe_balance.reindex(GENES).eq('balanced').to_numpy(),
    'probe_clean_both': (clean['mouse'].reindex(GENES.str.upper()).fillna(False).to_numpy(bool)
                         & clean['human'].reindex(human_symbol.str.upper()).fillna(False).to_numpy(bool)),
    'housekeeping': human_symbol.str.upper().isin(housekeeping).to_numpy(), 'non_one_to_one_pair': GENES.isin(dropped_pairs),
    'census_mouse_status': audit[audit.species.eq('mouse')].set_index('column').status.reindex(GENES).fillna('absent').to_numpy(),
    'census_human_status': audit[audit.species.eq('human')].set_index('column').status.reindex(GENES).fillna('absent').to_numpy(),
}, index=GENES)
LEVEL_INPUTS = pd.Series({'units': len(structure_ids), **{f'{s} units': int((specimen == s).sum()) for s in MOUSE + HUMAN},
                          'genes': len(GENES), 'pathways': len(GENE_SETS), 'nearly absent in one species': int(INFO.nearly_absent.sum()),
                          'zero counts in a specimen': int(INFO.zero_in_a_specimen.sum()), 'equal probe counts': int(INFO.equal_probes.sum()),
                          'equal probe counts and probe-clean in both': int((INFO.equal_probes & INFO.probe_clean_both).sum()),
                          'housekeeping (Eisenberg & Levanon 2013)': int(INFO.housekeeping.sum()),
                          'pairs inside non-1:1 orthogroups (notebook 64)': int(INFO.non_one_to_one_pair.sum())}, name='value')
LEVEL_INPUTS.to_csv(level_tables / 'inputs.csv')
display(LEVEL_INPUTS.to_frame())


def read_obs_column(group_, name):
    item = group_[name]
    if isinstance(item, h5py.Group):
        categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
        return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
    values = item[:]
    return values.astype(str) if values.dtype.kind in 'SO' else values


def lake_cortex_pseudobulk():
    """Raw-count sums of Lake's normal cortex PT nuclei per donor x sex x segment, over all genes (notebook 68)."""
    with h5py.File(LAKE, 'r') as handle:
        obs = pd.DataFrame({c: np.asarray(read_obs_column(handle['obs'], c)).astype(str)
                            for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(read_obs_column(handle['raw/var'], 'feature_name')).astype(str)
        keep = (obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3']) & obs.disease.eq('normal')
                & obs.region.eq('C') & obs.tissue.eq('cortex of kidney'))
        label = (obs.donor_id + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups_, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups_), len(var_ids), block_rows=5000)
    return {'groups': groups_.astype(str), 'var_ids': var_ids, 'var_names': var_names, 'sums': sums, 'totals': totals,
            'nuclei': nuclei}


lake = cached('lake_cortex_pt', lake_cortex_pseudobulk,
              params={'labels': 'PT-S1,PT-S2,PT-S3', 'disease': 'normal', 'region': 'C, cortex of kidney', 'group': 'donor|sex|segment'},
              inputs={'lake_file': [LAKE.name, LAKE.stat().st_size]})
lake68 = stored(nb68, 'lake_cortex_pt')        # guard: notebook 68's pseudobulk
for key in ('groups', 'var_ids', 'sums', 'totals', 'nuclei'):
    np.testing.assert_array_equal(np.asarray(lake[key]), np.asarray(lake68[key]))
guard_log.append("level control: the Lake cortex pseudobulk equals notebook 68's cache")
lake_groups = pd.DataFrame([g.split('|') for g in lake['groups'].astype(str)], columns=['donor', 'sex', 'segment'])
lake_groups['nuclei'] = lake['nuclei']
handle = ad.read_h5ad(NATIVE['HUK1_COR1'], backed='r')
human_panel_ids = handle.var['gene_ids'].astype(str)
handle.file.close()
by_id = dict(zip(orthologs.human_symbol.map(human_panel_ids[~human_panel_ids.index.duplicated()]), orthologs.mouse_symbol))
by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(lake['var_ids'].astype(str), lake['var_names'].astype(str))])
lake_mapped = target != ''
lake_on_genes = (pd.DataFrame(lake['sums'][:, lake_mapped], columns=target[lake_mapped]).T.groupby(level=0).sum().T
                 .reindex(columns=GENES, fill_value=0.0))
INFO['lake_mapped'] = GENES.isin(set(target[lake_mapped]))
nuclei = lake_groups.pivot(index='donor', columns='segment', values='nuclei').reindex(columns=list(SEGMENTS)).fillna(0)
lake_donors = sorted(nuclei.index[(nuclei >= MIN_NUCLEI).all(axis=1)])
assert lake_donors == sorted(LAKE_DONORS), lake_donors
lake_sex = lake_groups.drop_duplicates('donor').set_index('donor').sex
lake_row = {(dn, s): k for k, (dn, s) in enumerate(zip(lake_groups.donor, lake_groups.segment))}


def lake_block(donors, matrix):
    return np.stack([np.stack([np.asarray(matrix[lake_row[(dn, s)]], float) for s in SEGMENTS]) for dn in donors])


LAKE_SUMS = lake_block(lake_donors, lake_on_genes.to_numpy())
LAKE_ENSEMBL = lake_block(lake_donors, lake['sums'])
LAKE_IDS = pd.Index(lake['var_ids'].astype(str))
lake_male = np.array([lake_sex[dn] == 'male' for dn in lake_donors])


def census_table(label, suffix=''):
    return (pd.read_csv(CENSUS / f'{label}_pt_segment_pseudobulk_counts{suffix}.csv.gz', index_col=0),
            pd.read_csv(CENSUS / f'{label}_pt_segment_pseudobulk_meta{suffix}.csv', index_col=0))


def census_donors(meta_, segment_map, *, sex=None, stages=None):
    m = meta_[~meta_.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    m['segment'] = m.cell_type.map(segment_map)
    m = m[m.segment.notna() & m.n_cells.ge(MIN_NUCLEI)]
    if sex is not None:
        m = m[m.sex.eq(sex)]
    if stages is not None:
        m = m[m.development_stage.isin(stages)]
    complete = m.groupby('donor_id').segment.apply(lambda s: set(s) == set(segment_map.values()))
    return m[m.donor_id.isin(complete[complete].index)]


def census_block(table, m, donors, segments):
    group_of = pd.Series(m.index, index=pd.MultiIndex.from_arrays([m.donor_id, m.segment]))
    assert group_of.index.is_unique
    return np.stack([table.loc[[group_of[(dn, s)] for s in segments]].to_numpy(float) for dn in donors])


mouse_counts, mouse_meta = census_table('mouse')
human_counts_c, human_meta_c = census_table('human')
family_counts, family_meta = census_table('mouse', '_families')
mouse_adult = census_donors(mouse_meta, CENSUS_SEGMENTS, sex='male', stages=ADULT_STAGES)
mouse_males = census_donors(mouse_meta, CENSUS_SEGMENTS, sex='male')
census_human = census_donors(human_meta_c, CENSUS_HUMAN_SEGMENTS)
ADULT, MALES, CENSUS_HUMAN = (sorted(m.donor_id.unique()) for m in (mouse_adult, mouse_males, census_human))
assert (len(ADULT), len(MALES), len(CENSUS_HUMAN)) == (N_ADULT_MALES, N_MALES, N_CENSUS_HUMAN)
pd.testing.assert_frame_equal(family_meta.loc[mouse_meta.index], mouse_meta)
shared = sorted(set(family_counts.columns) & set(mouse_counts.columns))
differ = [g for g in shared if not np.array_equal(family_counts.loc[mouse_meta.index, g].to_numpy(), mouse_counts.loc[mouse_meta.index, g].to_numpy())]
assert set(differ) <= census_other['mouse'], differ
mouse_on_genes = mouse_counts.reindex(columns=GENES).fillna(0.0)
human_c_on_genes = human_counts_c.reindex(columns=GENES).fillna(0.0)
MOUSE_ADULT_SUMS = census_block(mouse_on_genes, mouse_adult, ADULT, SEGMENTS)
MOUSE_MALE_SUMS = census_block(mouse_on_genes, mouse_males, MALES, SEGMENTS)
CENSUS_HUMAN_SUMS = census_block(human_c_on_genes, census_human, CENSUS_HUMAN, ('PCT', 'S3'))
INFO['in_census_mouse'] = GENES.isin(mouse_counts.columns) & ~GENES.isin(census_other['mouse'])
INFO['in_census_human'] = GENES.isin(human_counts_c.columns) & ~GENES.isin(census_other['human'])
ANALYSED = (INFO.lake_mapped & INFO.in_census_mouse).to_numpy()
ANALYSED_CENSUS_HUMAN = (INFO.in_census_human & INFO.in_census_mouse).to_numpy()


def atlas_comparison(human_sums, mouse_sums, mask, *, human_weights=None):
    idx = np.flatnonzero(mask)
    h, m = human_sums[:, :, idx], mouse_sums[:, :, idx]
    totals_h, totals_m = h.sum(axis=2), m.sum(axis=2)
    values = np.vstack([equal_segment_log_cpm(h, totals_h, weights=human_weights), equal_segment_log_cpm(m, totals_m)])
    case = np.r_[np.ones(len(h), bool), np.zeros(len(m), bool)]
    fit = limma_trend_species(values, case, genes=GENES[idx])
    return {'fit': fit.reindex(GENES), 'human_totals': totals_h, 'mouse_totals': totals_m, 'centre': float(np.median(fit.coefficient))}


ATLAS = {'primary': atlas_comparison(LAKE_SUMS, MOUSE_ADULT_SUMS, ANALYSED),
         'all males': atlas_comparison(LAKE_SUMS, MOUSE_MALE_SUMS, ANALYSED),
         'Lake men': atlas_comparison(LAKE_SUMS[lake_male], MOUSE_ADULT_SUMS, ANALYSED),
         'Census human': atlas_comparison(CENSUS_HUMAN_SUMS, MOUSE_ADULT_SUMS, ANALYSED_CENSUS_HUMAN, human_weights=CENSUS_HUMAN_WEIGHTS)}
A = ATLAS['primary']['fit'].centred.to_numpy()
A_Q = ATLAS['primary']['fit'].q.to_numpy()


def atlas_strong(fit):
    a, q = fit.centred.to_numpy(), fit.q.to_numpy()
    return np.isfinite(a) & (q <= ATLAS_FDR) & (np.abs(a) >= ATLAS_LFC)


STRONG = atlas_strong(ATLAS['primary']['fit'])
ATLAS_TABLE = pd.DataFrame([{'atlas': name, 'genes': int(item['fit'].centred.notna().sum()), 'centre (log2)': item['centre'],
                             'atlas-strong': int(atlas_strong(item['fit']).sum())} for name, item in ATLAS.items()])
ATLAS_TABLE.to_csv(level_tables / 'atlas_fits.csv', index=False)
display(ATLAS_TABLE)

# our level
stats = MODEL.statistics
CONVERGED, SEPARATED = stats.converged.to_numpy(), stats.separated.to_numpy()
REF = CONVERGED & ~SEPARATED
L = grid_centred_level(pd.DataFrame(MODEL.delta, index=GENES), reference=REF).to_numpy()
L[~CONVERGED] = np.nan
L71 = g3.L.reindex(GENES)
both_l = np.isfinite(L) & L71.notna().to_numpy()
np.testing.assert_allclose(L[both_l], L71.to_numpy()[both_l], rtol=1e-9, atol=1e-9)
guard_log.append(f"level control: L equals notebook 71's S3 level on {int(both_l.sum()):,} shared genes")
valid = {n: (MODEL.specimen_converged[n] & ~MODEL.specimen_separated[n]).to_numpy() & CONVERGED for n in MOUSE + HUMAN}
PAIRS = pair_levels(MODEL.specimen_curves, [(h, m) for h in HUMAN for m in MOUSE], valid=valid, reference=REF, index=GENES)
PAIR_SUMMARY = pair_summary(PAIRS)
SAME = pair_levels(MODEL.specimen_curves, [tuple(MOUSE), tuple(HUMAN)], valid=valid, reference=REF, index=GENES)
UNITS_P, P_SUMS, P_TOTALS = segment_pseudobulk(counts, library, specimen, segment, segments=SEGMENTS)
CASE_UNITS = np.isin(UNITS_P, HUMAN)
P_RAW = species_difference(equal_segment_log_cpm(P_SUMS, P_TOTALS), CASE_UNITS)
P_CENTRE = float(np.median(P_RAW[REF]))
P = P_RAW - P_CENTRE
P[~CONVERGED] = np.nan
DIRECTION = np.where(SEPARATED, P, L)
MAGNITUDE = np.abs(DIRECTION)
GENE_TABLE = pd.concat([pd.DataFrame({
    'L': L, 'P': P, 'direction': DIRECTION, 'converged': CONVERGED, 'separated': SEPARATED, 'reference': REF,
    'T_spatial': stats.T_spatial.to_numpy(), 'A': A, 'A_q': A_Q, 'atlas_strong': STRONG,
    'A_all_males': ATLAS['all males']['fit'].centred.to_numpy(), 'A_lake_men': ATLAS['Lake men']['fit'].centred.to_numpy(),
    'A_census_human': ATLAS['Census human']['fit'].centred.to_numpy()}, index=GENES),
    PAIRS, PAIR_SUMMARY, SAME.add_prefix('same species: '), INFO], axis=1)
GENE_TABLE.to_csv(level_tables / 'gene_level.csv')

# bands
HK = INFO.housekeeping.to_numpy()
BANDS = {'L, housekeeping': (L, HK & REF), 'P, housekeeping': (P, HK & REF), 'A, housekeeping': (A, HK & ANALYSED & np.isfinite(A)),
         'mouse − mouse, all genes': (SAME[f'{MOUSE[0]} - {MOUSE[1]}'].to_numpy(), REF),
         'section − section, all genes': (SAME[f'{HUMAN[0]} - {HUMAN[1]}'].to_numpy(), REF), 'L, all genes': (L, REF)}
BAND_TABLE = pd.DataFrame([{'band': name, 'genes': int((mask & np.isfinite(values)).sum()),
                            'low': band(values[mask], BAND_QUANTILES)[0], 'high': band(values[mask], BAND_QUANTILES)[1]}
                           for name, (values, mask) in BANDS.items()]).set_index('band')
BAND_TABLE['width'] = BAND_TABLE.high - BAND_TABLE.low
BAND_TABLE.to_csv(level_tables / 'bands.csv')
HK_BAND = {'L': tuple(BAND_TABLE.loc['L, housekeeping', ['low', 'high']]), 'P': tuple(BAND_TABLE.loc['P, housekeeping', ['low', 'high']])}
display(BAND_TABLE.round(3))
print(f'Level control inputs and atlas {time.perf_counter() - t0:.0f} s')

In [ ]:
# (i) literature controls
lit = pd.read_csv(LITERATURE)
lit['expected_sign'] = lit.expected.str.strip().str.lower().map(EXPECTED_SIGN)
assert lit.expected_sign.notna().all()
lit['verified_for_mouse'] = lit.direction_verified_for_mouse.str.strip().str.lower().eq('yes')
lit['kind'] = lit.kind.str.strip().str.lower()
component_of = {sp: inventory[sp][inventory[sp].ortholog_class.isin(FAMILY_CLASSES)]
                .assign(key=lambda t: t.gene.str.upper()).set_index('key').component for sp in ('mouse', 'human')}


def our_gene(mouse_symbol, human_symbol_):
    if mouse_symbol in GENES:
        return mouse_symbol
    mouse = to_mouse.get(str(human_symbol_).upper())
    return mouse if mouse in GENES else None


def route(row):
    if row.kind == 'pathway':
        return 'pathway', None
    gene = our_gene(str(row.mouse_genes).strip(), str(row.human_genes).strip())
    if gene is not None:
        return 'gene', gene
    for sp, symbol in (('mouse', row.mouse_genes), ('human', row.human_genes)):
        component = component_of[sp].get(str(symbol).strip().upper())
        if component is not None:
            return 'family', int(component)
    return 'not evaluable', None


routes = [route(row) for row in lit.itertuples()]
lit['route'], lit['key'] = [r for r, _ in routes], [k for _, k in routes]
gene_rows = lit[lit.route.eq('gene')].copy()
position_of = pd.Series(np.arange(len(GENES)), index=GENES)
k_ = position_of.reindex(gene_rows.key).to_numpy().astype(int)
gene_rows['separated'], gene_rows['converged'] = SEPARATED[k_], CONVERGED[k_]
gene_rows['L'], gene_rows['P'], gene_rows['value'] = L[k_], P[k_], DIRECTION[k_]
gene_rows['correct'] = np.sign(gene_rows.value) == gene_rows.expected_sign
gene_rows['clears_band'] = np.where(gene_rows.separated, clears_band(gene_rows.value, gene_rows.expected_sign, HK_BAND['P']),
                                    clears_band(gene_rows.value, gene_rows.expected_sign, HK_BAND['L']))
gene_rows['pairs_same_sign_as_expected'] = (PAIR_SUMMARY.pairs_same_sign.to_numpy()[k_]
                                            & (np.sign(PAIR_SUMMARY.pair_min.to_numpy()[k_]) == gene_rows.expected_sign))
for column in ('pair_min', 'pair_max'):
    gene_rows[column] = PAIR_SUMMARY[column].to_numpy()[k_]
for column in ('equal_probes', 'probe_clean_both', 'nearly_absent', 'mouse_probes', 'human_probes'):
    gene_rows[column] = INFO[column].to_numpy()[k_]
gene_rows['A'], gene_rows['A_q'] = A[k_], A_Q[k_]
gene_rows['atlas_sign_as_expected'] = np.sign(gene_rows.A) == gene_rows.expected_sign
gene_rows.to_csv(level_tables / 'literature_genes.csv', index=False)
lit.to_csv(level_tables / 'literature_routing.csv', index=False)


def criterion_share(correct, minimum):
    n, hit = int(len(correct)), int(np.sum(correct))
    share = hit / n if n else np.nan
    low, high = wilson_interval(hit, n)
    status = 'not evaluable' if n < minimum else ('pass' if share >= PASS_SHARE else 'fail')
    return {'n': n, 'agree': hit, 'share': share, 'wilson_low': low, 'wilson_high': high, 'status': status}


usable = gene_rows[gene_rows.converged & gene_rows.value.notna()]
evaluable = usable[usable.verified_for_mouse]
subsets = {'primary (criterion i)': evaluable, 'primary, equal probes': evaluable[evaluable.equal_probes],
           'primary, equal probes, probe-clean in both': evaluable[evaluable.equal_probes & evaluable.probe_clean_both],
           'primary, not nearly absent': evaluable[~evaluable.nearly_absent], 'primary, not separated': evaluable[~evaluable.separated],
           'not verified for mouse (reported only)': usable[~usable.verified_for_mouse]}
LIT_TABLE = pd.DataFrame({name: criterion_share(rows.correct, MIN_LITERATURE) for name, rows in subsets.items()}).T
LIT_TABLE['clear the housekeeping band'] = [int(rows.clears_band.sum()) for rows in subsets.values()]
LIT_TABLE['four pairs share the expected sign'] = [int(rows.pairs_same_sign_as_expected.sum()) for rows in subsets.values()]
LIT_TABLE['atlas A has the expected sign'] = [int(rows.atlas_sign_as_expected.sum()) for rows in subsets.values()]
LIT_TABLE.loc['not verified for mouse (reported only)', 'status'] = 'reported only'
CRITERION_I = LIT_TABLE.loc['primary (criterion i)', ['n', 'agree', 'share', 'wilson_low', 'wilson_high', 'status']].to_dict()
LIT_TABLE.to_csv(level_tables / 'criterion_i.csv')
display(LIT_TABLE)


# (ii) and (iii)
def agreement(direction, magnitude, fit, mask, level_min=LEVEL_MIN):
    a = fit.centred.to_numpy()
    use = atlas_strong(fit) & mask & np.isfinite(direction) & (magnitude >= level_min)
    return criterion_share(np.sign(direction[use]) == np.sign(a[use]), MIN_ATLAS_GENES)


EQUAL_L, CLEAN_BOTH = INFO.equal_probes.to_numpy(), INFO.probe_clean_both.to_numpy()
ABSENT_, PAIRED_IN_FAMILY = INFO.nearly_absent.to_numpy(), INFO.non_one_to_one_pair.to_numpy()
primary_fit = ATLAS['primary']['fit']
AGREEMENT = pd.DataFrame({
    'primary (criterion ii)': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED),
    'all atlas-strong genes, any |L|': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED, level_min=0),
    'P in place of L': agreement(P, np.abs(P), primary_fit, CONVERGED),
    'equal probes': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED & EQUAL_L),
    'equal probes, probe-clean in both': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED & EQUAL_L & CLEAN_BOTH),
    'without nearly absent genes': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED & ~ABSENT_),
    'without separated genes': agreement(DIRECTION, MAGNITUDE, primary_fit, REF),
    'without pairs in non-1:1 orthogroups': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED & ~PAIRED_IN_FAMILY),
    'atlas: all 12 Census males': agreement(DIRECTION, MAGNITUDE, ATLAS['all males']['fit'], CONVERGED),
    "atlas: Lake's 3 men": agreement(DIRECTION, MAGNITUDE, ATLAS['Lake men']['fit'], CONVERGED),
    'atlas: Census human for Lake': agreement(DIRECTION, MAGNITUDE, ATLAS['Census human']['fit'], CONVERGED)}).T
CRITERION_II = AGREEMENT.loc['primary (criterion ii)'].to_dict()
AGREEMENT.to_csv(level_tables / 'criterion_ii.csv')
display(AGREEMENT)
with np.errstate(divide='ignore', invalid='ignore'):
    probe_ratio = np.log2(INFO.human_probes.to_numpy(float) / INFO.mouse_probes.to_numpy(float))
both_atlas = ANALYSED & ANALYSED_CENSUS_HUMAN
SPEARMAN_ROWS = {
    'L vs A (criterion iii)': (L, A, REF & ANALYSED), 'L vs A, equal probes': (L, A, REF & ANALYSED & EQUAL_L),
    'L vs A, equal probes and probe-clean in both': (L, A, REF & ANALYSED & EQUAL_L & CLEAN_BOTH),
    'L vs A, without nearly absent genes': (L, A, REF & ANALYSED & ~ABSENT_),
    'L vs A, without pairs in non-1:1 orthogroups': (L, A, REF & ANALYSED & ~PAIRED_IN_FAMILY), 'P vs A': (P, A, REF & ANALYSED),
    'L vs A (all 12 Census males)': (L, ATLAS['all males']['fit'].centred.to_numpy(), REF & ANALYSED),
    'L vs A (Census human)': (L, ATLAS['Census human']['fit'].centred.to_numpy(), REF & ANALYSED_CENSUS_HUMAN),
    'atlas ceiling: A (Lake) vs A (Census human)': (A, ATLAS['Census human']['fit'].centred.to_numpy(), both_atlas),
    'L vs P': (L, P, REF),
    'probe diagnostic: L - A vs log2(human / mouse probes), unequal probes': (L - A, probe_ratio, REF & ANALYSED & ~EQUAL_L & np.isfinite(probe_ratio))}
SPEARMAN = pd.DataFrame({name: bootstrap_spearman(x[mask], y[mask], n_boot=N_BOOT, seed=SEED)
                         for name, (x, y, mask) in SPEARMAN_ROWS.items()}).T
SPEARMAN.to_csv(level_tables / 'criterion_iii_spearman.csv')
display(SPEARMAN)

In [ ]:
# (iv) families, from the package's stage-01 native unit matrices
catalogue = pd.read_csv(nb62 / 'tables' / 'family_catalogue.csv')
for column in ('mouse n panel', 'human n panel', 'mouse total HCOP members', 'human total HCOP members'):
    catalogue[column] = catalogue[column].fillna(0).astype(int)
catalogue = catalogue.set_index('component')
catalogue['eligible'] = (catalogue['mouse n panel'].gt(0) & catalogue['human n panel'].gt(0)
                         & catalogue['mouse total HCOP members'].ne(catalogue['human total HCOP members']))
anchor_component = {name: int(inventory['mouse'].set_index('gene').component[anchor]) for name, anchor in PRENAMED.items()}
prenamed_of = {component: name for name, component in anchor_component.items()}
FAMILY_SET = sorted({int(c) for c in catalogue.index[catalogue.eligible]} | set(prenamed_of))
MEMBERS = {}
for sp in ('mouse', 'human'):
    table = inventory[sp][inventory[sp].ortholog_class.isin(FAMILY_CLASSES) & inventory[sp].component.isin(FAMILY_SET)]
    for variant, rows in (('all', table), ('clean', table[table.probe_clean.astype(bool)])):
        MEMBERS[(sp, variant)] = {int(c): sorted(g.gene_id.astype(str)) for c, g in rows.groupby('component')}
        MEMBERS[(sp, variant, 'symbols')] = {int(c): sorted(g.gene.astype(str)) for c, g in rows.groupby('component')}


def visium_family_sums():
    """Per specimen x segment family sums of raw counts from the native stage-01 matrices, and per-specimen detection."""
    out = {}
    for variant in ('all', 'clean'):
        out[f'sums_{variant}'] = np.zeros((len(UNITS_P), len(SEGMENTS), len(FAMILY_SET)))
        out[f'detected_{variant}'] = np.zeros((len(UNITS_P), len(FAMILY_SET)))
        out[f'available_{variant}'] = np.zeros((len(UNITS_P), len(FAMILY_SET)), bool)
    for u, name in enumerate(UNITS_P):
        species_ = 'human' if name in HUMAN else 'mouse'
        native = ad.read_h5ad(NATIVE[name])
        rows = pd.Index(structure_ids[specimen == name])
        if not rows.str.startswith(name + '_').all():
            raise ValueError(f'{name}: unit ids lack their specimen prefix.')
        local = rows.str.slice(len(name) + 1)
        if len(local.difference(native.obs_names)):
            raise ValueError(f'{name}: units are missing from the native matrix.')
        x = sparse.csr_matrix(native[local].X, dtype=float)
        if (x.data < 0).any() or not np.allclose(x.data, np.round(x.data)):
            raise ValueError(f'{name}: expected raw integer counts.')
        ids = native.var['gene_ids'].astype(str).to_numpy()
        seg = segment[specimen == name]
        for variant in ('all', 'clean'):
            members = MEMBERS[(species_, variant)]
            present = [c for c in FAMILY_SET if members.get(c)]
            sums, _ = family_sums(x, ids, {c: members[c] for c in present})
            columns = [FAMILY_SET.index(c) for c in present]
            for j, s in enumerate(SEGMENTS):
                out[f'sums_{variant}'][u, j, columns] = np.asarray(sums[seg == s].sum(axis=0)).ravel()
            out[f'detected_{variant}'][u, columns] = np.asarray((sums > 0).mean(axis=0)).ravel()
            out[f'available_{variant}'][u, columns] = True
    return out


# Guard (not cached): the native matrices hold the export's counts (shared genes, by Ensembl id)
export_ids = UNITS[hvm].var
for name in MOUSE + HUMAN:
    native = ad.read_h5ad(NATIVE[name])
    rows = native.obs_names.get_indexer(pd.Index(structure_ids[specimen == name]).str.slice(len(name) + 1))
    assert (rows >= 0).all(), f'{name}: units are missing from the native matrix'
    native_names = pd.Index(native.var_names.astype(str))
    duplicated = native_names.duplicated(keep=False)
    column_of = {n: i for i, n in enumerate(native_names) if not duplicated[i]}
    probe_genes = GENES[:: max(1, len(GENES) // 200)]
    symbols = probe_genes if name in MOUSE else export_ids.human_symbol.reindex(probe_genes)
    pick = [(g, column_of[s]) for g, s in zip(probe_genes, symbols) if s in column_of]
    x_native = sparse.csr_matrix(native.X, dtype=float)[rows][:, [c for _, c in pick]]
    x_export = counts[specimen == name][:, GENES.get_indexer([g for g, _ in pick])]
    assert len(pick) > 100 and (x_native != sparse.csr_matrix(x_export, dtype=float)).nnz == 0, name
    del native
guard_log.append('level control: the stage-01 native matrices hold the export counts (sampled shared genes, every specimen)')

FAMILY_PAYLOAD = cached('visium_family_sums', visium_family_sums, params={'segments': SEGMENTS, 'classes': FAMILY_CLASSES},
                        inputs={'units': structure_ids, 'segment': segment, 'specimen': specimen,
                                'members': {f'{k[0]}|{k[1]}': {str(c): m for c, m in v.items()} for k, v in MEMBERS.items() if len(k) == 2},
                                'matrices': {s: [p.name, p.stat().st_size] for s, p in NATIVE.items()}})
F, DETECTED = {}, {}
for variant in ('all', 'clean'):
    f = species_difference(equal_segment_log_cpm(FAMILY_PAYLOAD[f'sums_{variant}'], P_TOTALS), CASE_UNITS) - P_CENTRE
    f[~FAMILY_PAYLOAD[f'available_{variant}'].all(axis=0)] = np.nan
    F[variant] = f
    detected = FAMILY_PAYLOAD[f'detected_{variant}'] >= FAMILY_DETECTION
    DETECTED[variant] = detected[CASE_UNITS].all(axis=0) | detected[~CASE_UNITS].all(axis=0)
fetched = pd.read_csv(CENSUS / 'mouse_family_genes_fetched.csv').set_index('gene').found.astype(bool)
mouse_family_block = census_block(family_counts, mouse_adult, ADULT, SEGMENTS)
family_columns = pd.Index(family_counts.columns)
atlas_human = np.full((len(lake_donors), len(SEGMENTS), len(FAMILY_SET)), np.nan)
atlas_mouse = np.full((len(ADULT), len(SEGMENTS), len(FAMILY_SET)), np.nan)
for j, c in enumerate(FAMILY_SET):
    h_idx = LAKE_IDS.get_indexer(MEMBERS[('human', 'all')].get(c, []))
    m_symbols = MEMBERS[('mouse', 'all', 'symbols')].get(c, [])
    m_idx = family_columns.get_indexer(m_symbols)
    if len(h_idx) and (h_idx >= 0).all():
        atlas_human[:, :, j] = LAKE_ENSEMBL[:, :, h_idx].sum(axis=2)
    if len(m_idx) and (m_idx >= 0).all() and fetched.reindex(m_symbols).fillna(False).all():
        atlas_mouse[:, :, j] = mouse_family_block[:, :, m_idx].sum(axis=2)
complete = np.isfinite(atlas_human).all(axis=(0, 1)) & np.isfinite(atlas_mouse).all(axis=(0, 1))
ATLAS_F = np.full(len(FAMILY_SET), np.nan)
if complete.any():
    h_values = equal_segment_log_cpm(atlas_human[:, :, complete], ATLAS['primary']['human_totals'])
    m_values = equal_segment_log_cpm(atlas_mouse[:, :, complete], ATLAS['primary']['mouse_totals'])
    ATLAS_F[complete] = h_values.mean(axis=0) - m_values.mean(axis=0) - ATLAS['primary']['centre']
FAMILY_TABLE = pd.DataFrame({
    'family': [prenamed_of.get(c, '') for c in FAMILY_SET],
    'mouse members': [', '.join(MEMBERS[('mouse', 'all', 'symbols')].get(c, [])) for c in FAMILY_SET],
    'human members': [', '.join(MEMBERS[('human', 'all', 'symbols')].get(c, [])) for c in FAMILY_SET],
    'n_mouse_hcop': catalogue.loc[FAMILY_SET, 'mouse total HCOP members'].to_numpy(),
    'n_human_hcop': catalogue.loc[FAMILY_SET, 'human total HCOP members'].to_numpy(),
    'eligible': catalogue.loc[FAMILY_SET, 'eligible'].to_numpy(), 'detected': DETECTED['all'],
    'F': F['all'], 'F_probe_clean': F['clean'], 'atlas_F': ATLAS_F, 'atlas_complete': complete}, index=pd.Index(FAMILY_SET, name='component'))
FAMILY_TABLE['expected_sign'] = np.sign(FAMILY_TABLE.n_human_hcop - FAMILY_TABLE.n_mouse_hcop)
FAMILY_TABLE['F_as_expected'] = np.sign(FAMILY_TABLE.F) == FAMILY_TABLE.expected_sign
FAMILY_TABLE['F_agrees_with_atlas'] = np.sign(FAMILY_TABLE.F) == np.sign(FAMILY_TABLE.atlas_F)
FAMILY_TABLE['clears_P_band'] = clears_band(FAMILY_TABLE.F, FAMILY_TABLE.expected_sign, HK_BAND['P'])
FAMILY_TABLE.to_csv(level_tables / 'families.csv')
use = FAMILY_TABLE.eligible & FAMILY_TABLE.detected & FAMILY_TABLE.F.notna()
use_clean = FAMILY_TABLE.eligible & DETECTED['clean'] & FAMILY_TABLE.F_probe_clean.notna()
use_atlas = use & FAMILY_TABLE.atlas_complete
FAMILY_SUMMARY = pd.DataFrame({
    'F in the expected direction': criterion_share(FAMILY_TABLE.F_as_expected[use], 1),
    'F (probe-clean members) in the expected direction': criterion_share((np.sign(FAMILY_TABLE.F_probe_clean) == FAMILY_TABLE.expected_sign)[use_clean], 1),
    'mouse-expanded: F < 0': criterion_share(FAMILY_TABLE.F_as_expected[use & FAMILY_TABLE.expected_sign.lt(0)], 1),
    'human-expanded: F > 0': criterion_share(FAMILY_TABLE.F_as_expected[use & FAMILY_TABLE.expected_sign.gt(0)], 1),
    'atlas F in the expected direction': criterion_share((np.sign(FAMILY_TABLE.atlas_F) == FAMILY_TABLE.expected_sign)[use_atlas], 1),
    'F agrees in sign with atlas F': criterion_share(FAMILY_TABLE.F_agrees_with_atlas[use_atlas], 1)}).T.drop(columns='status')
FAMILY_SPEARMAN = bootstrap_spearman(FAMILY_TABLE.F[use_atlas], FAMILY_TABLE.atlas_F[use_atlas], n_boot=N_BOOT, seed=SEED)
FAMILY_SUMMARY.to_csv(level_tables / 'families_summary.csv')
display(FAMILY_SUMMARY)
print(f"Spearman F vs atlas F over {FAMILY_SPEARMAN['n']} complete families: {FAMILY_SPEARMAN['rho']:.2f} "
      f"[{FAMILY_SPEARMAN['low']:.2f}, {FAMILY_SPEARMAN['high']:.2f}]")
display(FAMILY_TABLE[FAMILY_TABLE.family.ne('')].set_index('family')[
    ['n_mouse_hcop', 'n_human_hcop', 'detected', 'F', 'F_probe_clean', 'atlas_F', 'F_as_expected', 'clears_P_band']].round(2))
family_rows = lit[lit.route.eq('family')].copy()
family_rows['F'] = FAMILY_TABLE.F.reindex(family_rows.key.astype(int)).to_numpy()
family_rows['correct'] = np.sign(family_rows.F) == family_rows.expected_sign
family_rows.to_csv(level_tables / 'literature_families.csv', index=False)

# pathways (descriptive)
pathway_rows = []
for pathway_id, members in GENE_SETS.items():
    idx = position_of.reindex(list(members)).dropna().astype(int).to_numpy()
    ours, atlas_ = idx[REF[idx]], idx[ANALYSED[idx] & np.isfinite(A[idx])]
    pathway_rows.append({'pathway_id': pathway_id, 'members': len(idx), 'members_L': len(ours),
                         'mean_L': L[ours].mean() if len(ours) else np.nan, 'members_A': len(atlas_),
                         'mean_A': A[atlas_].mean() if len(atlas_) else np.nan})
PATHWAYS_L = pd.DataFrame(pathway_rows).set_index('pathway_id')
PATHWAYS_L['rank_L'] = PATHWAYS_L.mean_L.rank(ascending=False)
PATHWAYS_L.to_csv(level_tables / 'pathways.csv')
both_sides = PATHWAYS_L.members_L.ge(5) & PATHWAYS_L.members_A.ge(5)
PATHWAY_SPEARMAN = bootstrap_spearman(PATHWAYS_L.mean_L[both_sides], PATHWAYS_L.mean_A[both_sides], n_boot=N_BOOT, seed=SEED)
literature_pathways = lit[lit.route.eq('pathway')].copy()
pathway_values = []
for row in literature_pathways.itertuples():
    mouse_list = [g.strip() for g in str(row.mouse_genes).split(';')]
    human_list = [g.strip() for g in str(row.human_genes).split(';')]
    pairs_ = zip(mouse_list, human_list) if len(mouse_list) == len(human_list) else ((g, '') for g in mouse_list)
    ours_ = sorted({g for g in (our_gene(m, h) for m, h in pairs_) if g is not None})
    idx = position_of.reindex(ours_).to_numpy().astype(int)
    on_ref, on_atlas = idx[REF[idx]], idx[ANALYSED[idx] & np.isfinite(A[idx])]
    mean_l = L[on_ref].mean() if len(on_ref) else np.nan
    pathway_values.append({'members': len(mouse_list), 'members_in_universe': len(idx), 'members_L': len(on_ref), 'mean_L': mean_l,
                           'mean_A': A[on_atlas].mean() if len(on_atlas) else np.nan,
                           'rank_among_pathways': int(1 + (PATHWAYS_L.mean_L > mean_l).sum()) if len(on_ref) else np.nan,
                           'members_L_with_expected_sign': int((np.sign(L[on_ref]) == row.expected_sign).sum())})
literature_pathways = pd.concat([literature_pathways.reset_index(drop=True), pd.DataFrame(pathway_values)], axis=1)
literature_pathways['correct'] = np.sign(literature_pathways.mean_L) == literature_pathways.expected_sign
literature_pathways.to_csv(level_tables / 'literature_pathways.csv', index=False)
print(f"Pathway means, L vs A over {PATHWAY_SPEARMAN['n']} pathways: Spearman {PATHWAY_SPEARMAN['rho']:.2f} "
      f"[{PATHWAY_SPEARMAN['low']:.2f}, {PATHWAY_SPEARMAN['high']:.2f}]")
display(literature_pathways[['control_id', 'expected', 'members_in_universe', 'mean_L', 'rank_among_pathways', 'correct']])

# criteria and reading (68's table, verbatim)
READINGS = {
    ('pass', 'pass'): 'Level is a usable descriptive measurement.',
    ('fail', 'pass'): 'Level agrees with the atlas genome-wide but not with the curated controls, which are protein '
                      'measurements. List the failing controls, their atlas sign and their probe balance.',
    ('pass', 'fail'): "The agreement rests on a few curated genes and does not hold genome-wide. Level stays out of the "
                      "paper's claims.",
    ('fail', 'fail'): "Level differences in our data are not interpretable. The paper keeps the SI's statement."}
CRITERIA = pd.DataFrame({'(i) literature controls, correct sign': CRITERION_I,
                         '(ii) atlas-strong genes with |L| >= 1, sign agreement': CRITERION_II}).T
CRITERIA['threshold'] = PASS_SHARE
CRITERIA['minimum n'] = [MIN_LITERATURE, MIN_ATLAS_GENES]
OUTCOME = tuple('pass' if s == 'pass' else 'fail' for s in CRITERIA.status)
CRITERIA.to_csv(level_tables / 'criteria.csv')
FAILING = evaluable[~evaluable.correct][['control_id', 'key', 'expected', 'confidence', 'value', 'pair_min', 'pair_max', 'A', 'A_q',
                                        'atlas_sign_as_expected', 'mouse_probes', 'human_probes', 'equal_probes', 'nearly_absent']].sort_values('value')
FAILING.to_csv(level_tables / 'failing_literature_controls.csv', index=False)
display(CRITERIA)
print(f'Reading ({OUTCOME[0]}, {OUTCOME[1]}): {READINGS[OUTCOME]}')
display(FAILING.round(2))
# 68's section 12 checks, re-reported as descriptive
post_controls = evaluable.assign(
    L_sign_as_atlas=np.sign(evaluable.value) == np.sign(evaluable.A),
    group=np.where(evaluable.correct, 'correct sign', np.where(evaluable.atlas_sign_as_expected, 'opposite sign; the atlas agrees with the control',
                                                               'opposite sign; the atlas also disagrees with the control')))
POST_HOC_CONTROLS = post_controls.groupby('group').agg(controls=('key', 'size'), same_sign_as_atlas=('L_sign_as_atlas', 'sum'),
                                                       genes=('key', lambda g: ', '.join(sorted(g))))
POST_HOC_CONTROLS.loc['all primary controls'] = [len(post_controls), int(post_controls.L_sign_as_atlas.sum()), '']
POST_HOC_CONTROLS.to_csv(level_tables / 'controls_vs_atlas.csv')
display(POST_HOC_CONTROLS)

In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(180 * MM, 200 * MM))
layout = GridSpec(2, 2, figure=fig, height_ratios=[1.15, 1], wspace=.3, hspace=.3)
LIMIT = 15
ax = fig.add_subplot(layout[0, 0])
shown = evaluable.sort_values('value').reset_index(drop=True)
ax.axvspan(*HK_BAND['L'], color=LIGHT, alpha=.8, lw=0, label='housekeeping band (L)')
ax.axvline(0, color=MUTED, lw=.5)
for i, r in shown.iterrows():
    ax.plot([r.pair_min, r.pair_max], [i, i], color=MUTED, lw=.8, solid_capstyle='round')
    ax.scatter(r.value, i, s=16, color=AGREE_COLOR if r.correct else DISAGREE_COLOR, marker='o' if r.correct else 'X', linewidths=0, zorder=3)
ax.set_yticks(range(len(shown)), [f"{r.key} ({'h' if r.expected_sign > 0 else 'm'})" for r in shown.itertuples()])
ax.set_ylim(-.8, len(shown) - .2)
ax.set_xlabel('Our level L (log2, centred)')
ax.scatter([], [], s=16, color=AGREE_COLOR, marker='o', label='expected sign')
ax.scatter([], [], s=16, color=DISAGREE_COLOR, marker='X', label='opposite sign')
ax.plot([], [], color=MUTED, lw=.8, label='range over the 4 pairs')
ax.legend(loc='upper left')
ax.set_title(f"a  Literature controls: {CRITERION_I['agree']}/{CRITERION_I['n']} correct", loc='left')
ax = fig.add_subplot(layout[0, 1])
base = REF & ANALYSED & np.isfinite(A)
decide = STRONG & CONVERGED & (MAGNITUDE >= LEVEL_MIN) & np.isfinite(DIRECTION)
agree = decide & (np.sign(DIRECTION) == np.sign(A))
x_shown, y_shown = np.clip(A, -LIMIT, LIMIT), np.clip(DIRECTION, -LIMIT, LIMIT)
ax.scatter(x_shown[base & ~STRONG], y_shown[base & ~STRONG], s=1.5, color=GREY, alpha=.3, linewidths=0, rasterized=True, label='analysed genes')
ax.scatter(x_shown[base & STRONG & ~decide], y_shown[base & STRONG & ~decide], s=2.5, color=MUTED, alpha=.6, linewidths=0, rasterized=True,
           label='atlas-strong, |L| < 1')
ax.scatter(x_shown[agree], y_shown[agree], s=4, color=AGREE_COLOR, linewidths=0, rasterized=True, label='atlas-strong, |L| ≥ 1: same sign')
ax.scatter(x_shown[decide & ~agree], y_shown[decide & ~agree], s=7, color=DISAGREE_COLOR, marker='X', linewidths=0, rasterized=True,
           label='atlas-strong, |L| ≥ 1: opposite sign')
ax.set_xlim(-LIMIT - .5, LIMIT + .5)
ax.set_ylim(-LIMIT - .5, LIMIT + .5)
ax.axhline(0, color=MUTED, lw=.5, zorder=0)
ax.axvline(0, color=MUTED, lw=.5, zorder=0)
ax.set_xlabel('Atlas level A (log2, centred)')
ax.set_ylabel('Our level L (log2, centred)')
ax.set_title(f"b  (ii) {CRITERION_II['agree']:,}/{CRITERION_II['n']:,} same sign", loc='left')
ax.legend(loc='upper left', markerscale=2.5, fontsize=5)
panel_c = layout[1, 0].subgridspec(2, 2, width_ratios=[1.25, 1], height_ratios=[1.1, 1], hspace=.45)
ax = fig.add_subplot(panel_c[0, 1])
rows_c = AGREEMENT.iloc[::-1]
y = np.arange(len(rows_c))
ax.errorbar(rows_c.share.astype(float), y, xerr=[rows_c.share.astype(float) - rows_c.wilson_low.astype(float),
                                                 rows_c.wilson_high.astype(float) - rows_c.share.astype(float)],
            fmt='o', ms=3, color=INK, ecolor=MUTED, elinewidth=.8, capsize=0)
ax.axvline(PASS_SHARE, color=DISAGREE_COLOR, lw=.8, ls='--')
ax.set_yticks(y, [f'{name} (n = {int(n):,})' for name, n in zip(rows_c.index, rows_c.n)], fontsize=5)
ax.set_xlabel('(ii) share with the atlas sign')
ax.text(-1.35, 1.06, 'c  Sensitivities', transform=ax.transAxes, fontweight='bold', fontsize=6.5, va='bottom')
ax = fig.add_subplot(panel_c[1, 1])
rows_s = SPEARMAN.drop(index=[i for i in SPEARMAN.index if i.startswith('probe diagnostic')]).iloc[::-1]
y = np.arange(len(rows_s))
ax.errorbar(rows_s.rho.astype(float), y, xerr=[rows_s.rho.astype(float) - rows_s.low.astype(float),
                                               rows_s.high.astype(float) - rows_s.rho.astype(float)],
            fmt='o', ms=3, color=INK, ecolor=MUTED, elinewidth=.8, capsize=0)
ax.axvline(0, color=MUTED, lw=.5)
ax.set_yticks(y, [name.replace('atlas ceiling: ', 'ceiling: ') for name in rows_s.index], fontsize=5)
ax.set_xlabel('(iii) Spearman ρ (95% bootstrap interval)')
ax = fig.add_subplot(layout[1, 1])
shown_f = FAMILY_TABLE[use_atlas]
for sign_, color, marker, label in ((-1, DISAGREE_COLOR, 's', 'more members in mouse'), (1, AGREE_COLOR, 'o', 'more members in human')):
    part = shown_f[shown_f.expected_sign.eq(sign_)]
    ax.scatter(part.atlas_F, part.F, s=7, color=color, marker=marker, linewidths=0, alpha=.8, label=label)
for _, r in shown_f[shown_f.family.ne('')].iterrows():
    ax.annotate(r.family, (r.atlas_F, r.F), xytext=(3, 2), textcoords='offset points', fontsize=5, color=INK)
ax.axhline(0, color=MUTED, lw=.5)
ax.axvline(0, color=MUTED, lw=.5)
ax.set_xlabel('Atlas family level (log2, centred)')
ax.set_ylabel('Our family level F (log2, centred)')
expected_row = FAMILY_SUMMARY.loc['F in the expected direction']
ax.set_title(f"d  Families: {int(expected_row.agree)}/{int(expected_row.n)} as expected; ρ with atlas {FAMILY_SPEARMAN['rho']:.2f}", loc='left')
ax.legend(loc='lower right', fontsize=5)
save(fig, 'fig_level_positive_control')
plt.show()

## 8 · The method table, its figure, the comparison with the old numbers and the material-change rules

`method_table.csv` has one row per method × dataset: family, source (71 or 75), gene model, calls (redundancy
clusters), relabeled calls, tested sets, NCDR, calibration share, the pass verdict of section 2, the equal-probe calls
and exposure (human vs mouse), the Jaccard with the primary, the old number and what the row returns.

**The figure.** One row per method. Filled dots: the condition split's calls (two for the single-split clustering
row). Open dots: the two relabelings. The dashed line marks 5% of the tested sets. Right margin: NCDR and the
calibration share; a failed rule is named.

In [ ]:
ORDER = ['primary', 'raw_camerapr', 'raw_camerapr_ranks', 'raw_gsea_ranknormal', 'raw_gsea_raw', 'nb_joint', 's1_shape_gsea',
         'segment', 'segment_joint', 'deseq2_segments_gsea', 'deseq2_segments_ora', 'deseq2_wholept_gsea', 'deseq2_wholept_ora',
         's1_level_camerapr', 's1_level_gsea', 's1_total_camerapr', 's1_total_gsea', 's3_level_ora', 'clustering',
         'clustering_single', 'modules_top10', 'modules_significant', 's2_shape', 's2_level']
OLD = {
    (hvm, 'primary'): '69 (SCF13): 97; 0, 0; NCDR 0; 0.044; pass', (aki, 'primary'): '69 (v5 DPT): 99; 16, 10; NCDR 0.13; 0.101; fail',
    (hvm, 'clustering'): '69: 132; 0, 0; pass', (aki, 'clustering'): '69: 455; 2, 0; pass',
    (hvm, 'clustering_single'): '69: 162, 46; 0, 0; pass', (aki, 'clustering_single'): '69: 465, 468; 2, 0; pass',
    (hvm, 'raw_camerapr'): '66 (SCF13): 98; 46, 29; NCDR 0.38; 0.121; fail', (aki, 'raw_camerapr'): '66 (v5): 144; 87, 86; NCDR 0.60; 0.170; fail',
    (hvm, 'raw_camerapr_ranks'): '66: 81; 28, 6; NCDR 0.21; 0.108; fail', (aki, 'raw_camerapr_ranks'): '66: 140; 69, 60; NCDR 0.46; 0.161; fail',
    (hvm, 'raw_gsea_ranknormal'): '66: 146; 75, 80; NCDR 0.53; 0.147; fail', (aki, 'raw_gsea_ranknormal'): '66: 203; 120, 146; NCDR 0.66; 0.212; fail',
    (hvm, 'raw_gsea_raw'): '66: 0; 51, 35; 0.176; fail', (aki, 'raw_gsea_raw'): '66: 0; 46, 5; 0.207; fail',
    (hvm, 'nb_joint'): '66: 32; 3, 0; NCDR 0.05; 0.067; pass', (aki, 'nb_joint'): '66: 12; 0, 0; NCDR 0; 0.072; pass',
    (hvm, 'segment'): 'different test: Gaussian coordinate-free step screen (45, 61): 50; 0, 0; pass',
    (aki, 'segment'): 'different test: Gaussian coordinate-free step screen (45, v5): 26; 12, 6; NCDR 0.35; fail',
    (hvm, 'segment_joint'): 'NB step joint test on SCF13 (65, A4·JNT): 62; 9, 9; 0.105 (Pearson VIF); 15; 1, 1; 0.079 (Gaussian VIF)',
    (hvm, 'deseq2_wholept_gsea'): '31/37 (SCF13 units): 158; 193, 255; NCDR 1.42; fail',
    (hvm, 'deseq2_segments_gsea'): '31/37: 204; 343, 255; NCDR 1.47; fail',
    (hvm, 'deseq2_wholept_ora'): '56: 80; 0, 1; pass (relabeling rule); equal-probe calls 27',
    (hvm, 'deseq2_segments_ora'): '56: 118; 0, 1; pass (relabeling rule); equal-probe calls 49, exposure 0.60',
    (hvm, 'modules_top10'): '56(c) (Gaussian, k = 8): 44; 23, 12; NCDR 0.40; fail',
    (hvm, 'modules_significant'): '61 B2 (Gaussian split-plot genes, k = 4): 23; 0, 0; pass'}

MT_ = pd.DataFrame(ROWS)
MT_['order'] = MT_.row.map({k: i for i, k in enumerate(ORDER)})
assert MT_.order.notna().all(), set(MT_.row) - set(ORDER)
overlap_index = OVERLAP.dropna(subset=['row']).set_index(['dataset', 'row'])
MT_['Jaccard with the primary (sets)'] = [overlap_index['Jaccard (sets)'].get((r.dataset, r.row), np.nan) for r in MT_.itertuples()]
MT_['Jaccard with the primary (redundancy clusters)'] = [overlap_index['Jaccard (redundancy clusters)'].get((r.dataset, r.row), np.nan)
                                                         for r in MT_.itertuples()]
probe_index = PROBE.set_index('row')
hvm_rows = MT_.dataset.eq(DATA[hvm]['title'])
MT_['equal-probe calls'] = np.where(hvm_rows, MT_.row.map(probe_index['equal-probe calls']), '')
MT_['probe exposure'] = np.where(hvm_rows, MT_.row.map(probe_index.exposure), np.nan)
title_to_ds = {DATA[ds]['title']: ds for ds in DATASETS}
MT_['old'] = [OLD.get((title_to_ds[r.dataset], r.row), '') for r in MT_.itertuples()]
METHOD_TABLE = MT_.sort_values(['dataset', 'order'], ascending=[False, True]).drop(columns='order').reset_index(drop=True)
METHOD_TABLE.to_csv(tables / 'method_table.csv', index=False)
with pd.option_context('display.max_colwidth', 60):
    for ds in DATASETS:
        print(DATA[ds]['title'])
        display(METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('method')[
            ['family', 'calls', 'relabeled calls', 'tested', 'NCDR', 'calibration share', 'relabeling rule', 'calibration', 'pass',
             'probe exposure', 'Jaccard with the primary (redundancy clusters)']].round(3))

fig, axes = plt.subplots(1, 2, figsize=(180 * MM, 120 * MM), sharey=True, gridspec_kw={'wspace': .95})
labels_order = [ROW_INFO[k]['label'] for k in ORDER]
for ax, ds in zip(axes, DATASETS):
    rows_ds = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('row')
    for i, key in enumerate(ORDER):
        y = len(ORDER) - 1 - i
        if key not in rows_ds.index:
            ax.text(0, y, 'not defined', fontsize=5, color=MUTED, va='center')
            continue
        r = rows_ds.loc[key]
        color = FAMILY_COLOR[r.family]
        condition_calls = [int(x) for x in str(r.calls).split(', ')]
        relabeled = [int(x) for x in str(r['relabeled calls']).split(', ')]
        ax.plot([0, max(condition_calls + relabeled)], [y, y], color=LIGHT, lw=.6, zorder=1)
        ax.scatter(relabeled, [y] * len(relabeled), s=18, facecolor='white', edgecolor=color, linewidth=1, zorder=3)
        ax.scatter(condition_calls, [y] * len(condition_calls), s=18, color=color, edgecolor='white', linewidth=.5, zorder=4)
        flags = [] if r['relabeling rule'] == 'pass' else ['rel.']
        flags += [] if r.calibration == 'pass' else ['cal.']
        ncdr = '—' if not np.isfinite(r.NCDR) else f'{r.NCDR:.2f}'
        ax.text(1.02, y, f"NCDR {ncdr} · cal. {r['calibration share']:.3f}" + (f" · fails {' + '.join(flags)}" if flags else ''),
                transform=ax.get_yaxis_transform(), va='center', fontsize=4.5, color=INK if flags else MUTED)
    ax.axvline(NULL_RATE * len(DATA[ds]['sets']), color=MUTED, lw=.6, ls='--', zorder=0)
    ax.set_xscale('symlog', linthresh=10)
    ax.set_xlim(left=-.5)
    ax.set_ylim(-.8, len(ORDER) - .3)
    ax.set_xlabel('sets called')
    ax.set_title(DATA[ds]['title'], loc='left')
axes[0].set_yticks(range(len(ORDER)), labels_order[::-1], fontsize=5)
for tick, key in zip(axes[0].get_yticklabels(), ORDER[::-1]):
    tick.set_color(FAMILY_COLOR[ROW_INFO[key]['family']])
handles = [plt.Line2D([], [], marker='o', lw=0, color=INK, markersize=4, label='condition split'),
           plt.Line2D([], [], marker='o', lw=0, markerfacecolor='white', markeredgecolor=INK, markersize=4, label='relabelings'),
           *[plt.Line2D([], [], marker='s', lw=0, color=c, markersize=4, label=f) for f, c in FAMILY_COLOR.items()]]
fig.legend(handles=handles, loc='upper center', ncol=7, bbox_to_anchor=(.5, .02), fontsize=5)
save(fig, 'fig_method_table')
plt.show()

### 8.1 · The old numbers beside the new ones (descriptive) and the material-change rules

`compare_old.csv` sets each row with an old result beside its new one. The unit, label, coordinate and offset changes
are notebook 71's (its D1–D8); the segment F-test is a new statistic, so its old row is a different test.

**Material changes** (protocol section 10; stated in the summary and the ledger):
- **M1.** cameraPR on rank-normal raw `T_spatial` passes the rule on either dataset (old: fails both parts on both).
- **M2.** The NB joint test changes verdict on either dataset (old: passes both).
- **M3.** A signed average-level GSEA row (whole PT or S1/S2/S3) passes on human vs mouse, or a DE + ORA row fails
  (old: fail, pass).
- **M4.** S1/S2/S3 DE + ORA is no longer probe-exposed (exposure < 0.5; old 0.60), or the primary becomes exposed.
- **M5.** The segment F-test fails the rule on human vs mouse.
- **M6.** The level control's reading changes: (i) or (ii) changes status (old: fail, pass), (iii) moves by more than
  0.10 (old 0.60), or the probe-leak Spearman falls below 0.2 (old 0.45).
- **M7.** The primary and any discrete or average-level row share more than 0.3 of their redundancy clusters
  (Jaccard); the segment F-test and its sensitivity are reported but not judged by M7.

In [ ]:
COMPARE = METHOD_TABLE[METHOD_TABLE.old.ne('')].assign(
    new=lambda t: [f"{r.calls}; {r['relabeled calls']}; NCDR {'—' if not np.isfinite(r.NCDR) else f'{r.NCDR:.2f}'}; "
                   f"{r['calibration share']:.3f}; {r['pass']}" for _, r in t.iterrows()])[['dataset', 'row', 'method', 'old', 'new']]
COMPARE.to_csv(tables / 'compare_old.csv', index=False)
with pd.option_context('display.max_colwidth', 110):
    display(COMPARE.set_index(['dataset', 'method']).drop(columns='row'))


def cell(ds, key, column):
    return METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title']) & METHOD_TABLE.row.eq(key)].iloc[0][column]


rho_iii = float(SPEARMAN.loc['L vs A (criterion iii)', 'rho'])
rho_probe = float(SPEARMAN.loc['probe diagnostic: L - A vs log2(human / mouse probes), unequal probes', 'rho'])
m7_rows = METHOD_TABLE[METHOD_TABLE.family.isin(['discrete', 'average-level']) & ~METHOD_TABLE.row.isin(['segment', 'segment_joint'])]
m7_max = m7_rows.loc[m7_rows['Jaccard with the primary (redundancy clusters)'].idxmax()]
MATERIAL = pd.DataFrame([
    {'rule': 'M1 · raw T_spatial cameraPR passes', 'old': 'fails on both datasets',
     'new': '; '.join(f"{DATA[ds]['title']}: {cell(ds, 'raw_camerapr', 'pass')}" for ds in DATASETS),
     'met': any(cell(ds, 'raw_camerapr', 'pass') == 'pass' for ds in DATASETS)},
    {'rule': 'M2 · NB joint test changes verdict', 'old': 'passes on both datasets',
     'new': '; '.join(f"{DATA[ds]['title']}: {cell(ds, 'nb_joint', 'pass')}" for ds in DATASETS),
     'met': any(cell(ds, 'nb_joint', 'pass') != 'pass' for ds in DATASETS)},
    {'rule': 'M3 · average-level GSEA passes, or DE + ORA fails (human vs mouse)', 'old': 'GSEA fail, fail; ORA pass, pass',
     'new': '; '.join(f"{k}: {cell(hvm, k, 'pass')} (relabeling {cell(hvm, k, 'relabeling rule')}, calibration {cell(hvm, k, 'calibration')})"
                      for k in ('deseq2_wholept_gsea', 'deseq2_segments_gsea', 'deseq2_wholept_ora', 'deseq2_segments_ora')),
     'met': (cell(hvm, 'deseq2_wholept_gsea', 'pass') == 'pass' or cell(hvm, 'deseq2_segments_gsea', 'pass') == 'pass'
             or cell(hvm, 'deseq2_wholept_ora', 'pass') == 'fail' or cell(hvm, 'deseq2_segments_ora', 'pass') == 'fail')},
    {'rule': 'M4 · S1/S2/S3 ORA no longer probe-exposed, or the primary exposed', 'old': 'ORA exposure 0.60',
     'new': f"S1/S2/S3 ORA exposure {probe_index.exposure['deseq2_segments_ora']:.2f}; primary {probe_index.exposure['primary']:.2f}",
     'met': bool(probe_index.exposure['deseq2_segments_ora'] < EXPOSED or probe_index.exposure['primary'] >= EXPOSED)},
    {'rule': 'M5 · the segment F-test fails on human vs mouse', 'old': 'Gaussian step screen 50; 0, 0 (different test)',
     'new': f"{cell(hvm, 'segment', 'calls')}; {cell(hvm, 'segment', 'relabeled calls')}; {cell(hvm, 'segment', 'calibration share'):.3f}; "
            f"{cell(hvm, 'segment', 'pass')}", 'met': cell(hvm, 'segment', 'pass') == 'fail'},
    {'rule': "M6 · the level control's reading changes", 'old': '(i) fail 17/27; (ii) pass 90.0%; (iii) 0.60; probe leak 0.45',
     'new': f"(i) {CRITERION_I['status']} {CRITERION_I['agree']}/{CRITERION_I['n']}; (ii) {CRITERION_II['status']} "
            f"{CRITERION_II['share']:.1%}; (iii) {rho_iii:.2f}; probe leak {rho_probe:.2f}",
     'met': bool(CRITERION_I['status'] != 'fail' or CRITERION_II['status'] != 'pass' or abs(rho_iii - .60) > .10 or rho_probe < .2)},
    {'rule': 'M7 · the primary shares > 0.3 of its redundancy clusters with a discrete or average-level row',
     'old': 'Jaccard over sets 0.07–0.17 (steps 0.75 excepted)',
     'new': f"largest: {m7_max.method} ({m7_max.dataset}) {m7_max['Jaccard with the primary (redundancy clusters)']:.2f}",
     'met': bool((m7_rows['Jaccard with the primary (redundancy clusters)'] > M7_LIMIT).any())}])
MATERIAL.to_csv(tables / 'material_change.csv', index=False)
with pd.option_context('display.max_colwidth', 160):
    display(MATERIAL)

## 9 · How the protocol was read

Implementation choices the protocol leaves open. Each was fixed in the code before the first full run and is not
tuned.
1. **Logic version** `75.package.1`, by the coordinator's instruction (the protocol named `75.methods.1`).
2. **3.3 · Designs.** `steps` is written as base + group × each segment, with no separate group column. That is the
   same column space as level + group × segment, so the per-segment group differences are coefficients. α is 71's
   condition-split α with its convergence flags. Non-converged genes are median-filled, as in 71.
3. **3.1 · GSEA rows.** A set that gseapy's size filter does not test is not counted. A missing nominal p counts as 1 in
   the calibration share.
4. **3.4 · DESeq2.**
   - The design factor is `condition` (case/reference) in place of notebook 31's `species`; the AKI rows use the
     same code.
   - Genes whose four pseudobulks total zero in a scope would be left out there; none was.
   - Signed GSEA calibration p per set: the nominal p (whole PT), or the smallest segment p × the number of segments
     in which the set was tested.
   - ORA runs over all 71 sets, with each scope's genes as background (notebook 56). "Tested" counts the sets with
     10–300 members among the genes that have a Wald stat.
5. **3.5 · Module lists** leave out genes whose fit did not converge or is separated, because their Δ(x) is not a
   shape. The top 10% is round(0.1 × G) genes by z among the eligible genes. Tested = all 71 sets.
6. **3.6 · The S3 row.** It has no calibration share in 71, so section 2's several-tests convention is used (best p ×
   2 directions). G4 checks its calls and relabeled calls only.
7. **Section 4 · Probe exposure.**
   - The joint test on equal-probe genes refits its VIF on the restricted relabelings.
   - The single-split clustering row pools its two runs.
   - Exposure uses condition-split calls only.
8. **Section 5 · Overlap.** The single-split clustering row is represented by its default-null row. The offset lean
   uses 71's S1 cameraPR level two-sided p and shape one-sided p, each turned into z = Φ⁻¹.
9. **Section 6 · Heatmaps.**
   - No segment-transition lines are drawn.
   - The bins span 71's grid range; units outside it fall into the end bins (notebook 61's rule).
   - The program with most calls is chosen (ties: lowest best FDR), the module with most enriched sets (ties: lower
     index), and the cluster with most called sets.
10. **Section 7 · The level control.**
    - Lake is mapped with the human panel's Ensembl ids from the stage-01 native HUK1_COR1 matrix (the same 10x
      feature table as notebook 01's). Duplicated native symbols are dropped from that id map.
    - The analysed genes are 71's universe in both atlases, so A is refitted on them.
    - The native-matrix guard samples about 200 shared genes per specimen.
11. **Families of rows.** The total arm (`T_total`) was labelled average-level before the run, although `T_total`
    contains the positional term. Section 2's rule does not depend on the label; M7 does (section 10).
12. **M3.** "A DE + ORA row fails" is judged by section 2's full rule, calibration included, and the failing part is
    stated.

## 10 · Summary

Two male control mice, two cortex sections of one male human donor and two AKI mice; specimens are the replicates and
every number is descriptive. Cells give calls; relabeled calls; calibration share; verdict under section 2's rule.

**Human vs mouse.**

| Row | Result | Old |
|---|---|---|
| Positional F-test (primary, 71) | 83; 1, 0; 0.071 · pass | 69: 97; 0, 0; 0.044 |
| Segment F-test (new, coordinate-free steps) | 76; 0, 0; 0.055 · pass | Gaussian step screen 50; 0, 0 (different test) |
| NB joint test on `T_spatial` | 52; 0, 2; 0.072 · pass | 66: 32; 3, 0; 0.067 |
| cameraPR on raw `T_spatial` | 115; 53, 50; 0.132 · fail | 66: 98; 46, 29; 0.121 · fail |
| GSEA on raw `T_spatial` (rank-normal) | 152; 95, 79; 0.156 · fail | 66: 146; 75, 80; 0.147 · fail |
| Whole-PT / S1/S2/S3 DESeq2 + signed GSEA | 146; 189, 261 / 197; 220, 286 · fail | 158; 193, 255 / 204; 343, 255 · fail |
| Whole-PT / S1/S2/S3 DESeq2 + ORA | 81; 0, 0; 0.012 / 164; 0, 1; 0.018 · pass | 80; 0, 1 / 118; 0, 1 · pass |
| Curve modules, top 10% / q ≤ 0.05 genes | 54; 0, 3 / 59; 0, 0 · pass | 56(c) 44; 23, 12 · fail / B2 23; 0, 0 · pass |
| Clustering (71) | 94; 0, 0 · pass | 69: 132; 0, 0 |

The level and total arms, and GSEA on the positional z, fail. ssGSEA passes but calls 866 (shape) and 1,208 (level) of
1,513 sets.

**AKI vs control.**
- Pass: only the DE + ORA rows (259; 0, 0 and 411; 0, 0), clustering (459; 0, 0), the q ≤ 0.05 module twin
  (307; 0, 4) and ssGSEA.
- The primary fails calibration (0.120).
- The NB joint test fails calibration: 16; 0, 0; 0.080.
- The segment F-test fails both parts: 38; 24, 17; 0.115.

**Material changes.**
- **M2 is met:** the NB joint test now fails calibration on AKI (0.080; old 0.072).
- **M7 is met through one row:** the total arm on AKI shares 0.42 of its redundancy clusters with the primary.
  `T_total` contains the positional term by construction. Every DE row and every level row stays at or below 0.23.
- M1, M3, M4, M5 and M6 are not met.

**Segment F-test.** It calls 76 sets on human vs mouse; 71 are also primary calls (Jaccard 0.81). The continuous test
adds 12, the steps add 5.

**Probe exposure (human vs mouse).**
- Positional rows: 0.14–0.25 (primary 0.17, segment F-test 0.20); GSEA on the positional z 0.39.
- DE rows: 0.63–0.73 (S1/S2/S3 ORA 0.71, old 0.60).
- Clustering: 0.69. Level list: 0.85.

**Offset lean (human vs mouse).** 72% of all sets lean to the offset. Against each DE row, none of the calls that
only the primary makes lean to the offset, against 95–97% of the calls that only the DE row makes.

**Level control** (reading unchanged: fail, pass): (i) 18 of 27 controls with the expected sign (fail); (ii) 89.8%
(pass); (iii) ρ 0.60 against an atlas ceiling of 0.97; families 48%; probe leak 0.46.

**Programs.** The 83 human vs mouse calls form 43 programs. Figure 4e shows "Peroxisomal Lipid Metabolism" (6 calls).

## 11 · Run record

In [ ]:
import platform

from rpy2.robjects import r as R

cache_entries = []
for sidecar in sorted(cache.glob('*.key.json'), key=lambda p: json.loads(p.read_text())['written_utc']):
    meta_ = json.loads(sidecar.read_text())
    if meta_['stage'] in SECONDS:
        cache_entries.append({'stage': meta_['stage'], 'key': meta_['key'], 'written_utc': meta_['written_utc'],
                              'compute seconds': round(SECONDS[meta_['stage']], 1)})
CACHE_TABLE = pd.DataFrame(cache_entries).drop_duplicates('stage', keep='last').sort_values('stage').reset_index(drop=True)
display(CACHE_TABLE)
record = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
    'protocol': 'docs/results/pt-package-pathway-methods.md (commit 07ef30c)',
    'inputs': {ds: PROVENANCE[ds] for ds in DATASETS}, 'datasets': INPUT_TABLE.to_dict(), 'n_jobs': N_JOBS, 'mt_genes': MT,
    'notebook_71': str(T71.relative_to(results_root)),
    'segment_ftest': {'module': 'pseudospace.discrete_comparators.segment_f_test', 'df': SEGMENT_DF,
                      'tables': [f'gene_statistics_segment_{ds}.csv' for ds in DATASETS] + [f'pathway_tests_segment_{ds}.csv' for ds in DATASETS],
                      'summary': SEGMENT_SUMMARY.to_dict(orient='records')},
    'method_table': METHOD_TABLE.drop(columns=['returns']).to_dict(orient='records'),
    'joint_test_vif': JOINT_FIT.to_dict(orient='records'), 'diagnostics_raw_tspatial': DIAGNOSTICS.to_dict(orient='records'),
    'modules_k': MODULE_K.to_dict(orient='records'), 'probe_exposure': PROBE.to_dict(orient='records'),
    'programs': PROGRAM_TABLE.to_dict(orient='records'), 'heatmaps': HEAT_CHOICES.to_dict(orient='records'),
    'level_control': {'criteria': CRITERIA.reset_index().to_dict(orient='records'), 'reading': READINGS[OUTCOME],
                      'spearman': SPEARMAN.reset_index().to_dict(orient='records'), 'families': FAMILY_SUMMARY.reset_index().to_dict(orient='records'),
                      'family_spearman': FAMILY_SPEARMAN, 'pathway_spearman': PATHWAY_SPEARMAN,
                      'bands': BAND_TABLE.reset_index().to_dict(orient='records')},
    'material_change': MATERIAL.to_dict(orient='records'), 'guards': guard_log,
    'versions': {'python': platform.python_version(), 'R': str(R('R.version.string')[0]),
                 'limma': str(R('as.character(packageVersion("limma"))')[0]),
                 **{m: version(m) for m in ('numpy', 'scipy', 'pandas', 'rpy2', 'joblib', 'matplotlib', 'gseapy', 'pydeseq2',
                                            'scikit-learn', 'pseudospace-reconstruction')}},
    'stage_cache': CACHE_TABLE.to_dict(orient='records'), 'compute_seconds_total': float(sum(SECONDS.values())),
    'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print('\n'.join('· ' + line for line in guard_log))
print(f"Wall time {record['wall_seconds']:.0f} s; summed uncached compute time of the cached stages {record['compute_seconds_total']:.0f} s")